# Sliced Bread: a personal budget tracker

Add your income and spending; the app shows how much is left to invest each month, warns you near and over
your limits, compares your split with a research-backed plan, and tracks a savings goal.

**To run it:** Runtime > Run all. The app opens at the bottom of this notebook, and a public `gradio.live`
link is printed above it (best for the class demo). If a button shows an error, run the last cell and share
what it prints.

Every source file is below as its own cell. Each cell starts with `%%writefile`, which saves the cell as
that `.py` file so the files can import each other. This notebook is generated from the files in
[github.com/sourypink/Sliced-Bread](https://github.com/sourypink/Sliced-Bread) by `build_notebook.py`.

In [ ]:
# Install the exact versions the app was tested with.
!pip -q install gradio==6.31.0 pytest==9.2.0

## `budget.py`

The math: categories and subcategories, totals, what is left, limit and target alerts, input checks. No Gradio here.

In [ ]:
%%writefile budget.py
"""Budget math for Sliced Bread: pure functions, no Gradio, all money in whole cents."""

import re
from datetime import date

# Money comes in as income and goes out into subcategories (Rent goes in Necessities). Every subcategory sits in
# one of four fixed categories, which the percentage targets and History are built on.
INCOME = "income"
CATEGORIES = ("needs", "wants", "investing", "savings")
CATEGORY_NAMES = {"needs": "Needs", "wants": "Wants", "investing": "Investing", "savings": "Savings"}
SPENDING_CATEGORIES = ("needs", "wants")  # what limits and "spent" cover; investing and savings are minimums
PRESET_SUBCATEGORIES = (
    {"key": "necessities", "name": "Necessities", "category": "needs"},
    {"key": "discretionary", "name": "Discretionary", "category": "wants"},
    {"key": "travel", "name": "Travel", "category": "wants"},
    {"key": "investments", "name": "Investments", "category": "investing"},
    {"key": "emergency_fund", "name": "Emergency fund", "category": "savings"},
)
PRESET_KEYS = tuple(preset["key"] for preset in PRESET_SUBCATEGORIES)
MAX_CUSTOM_SUBCATEGORIES = 8
ALL_SPENDING = "All spending"  # History's unzoomed view
TIERED_SECTION = "investments"  # its lines belong to a tier the user names, like "Retirement"
UNSAFE_KEY_CHARACTERS = re.compile(r"[^a-z0-9]+")

# Starting plans the user can pick at sign-up or from Goals, each with the research behind it.
PLAN_PRESETS = (
    {
        "key": "balanced", "name": "Balanced",
        "targets": {"needs": 50, "wants": 30, "investing": 15, "savings": 5},
        "why": "Half on must-haves, 15% for retirement, 5% for short-term savings.",
        "source": "Warren & Tyagi, All Your Worth (2005): 50/30/20. Fidelity's 50/15/5 saving guideline.",
    },
    {
        "key": "safety_net", "name": "Safety net first",
        "targets": {"needs": 50, "wants": 20, "investing": 10, "savings": 20},
        "why": "Build an emergency fund before investing hard.",
        "source": "Vanguard (2025, 12,400 investors): $2,000+ in emergency savings, 21% higher financial well-being.",
    },
    {
        "key": "invest_more", "name": "Invest more",
        "targets": {"needs": 45, "wants": 20, "investing": 30, "savings": 5},
        "why": "A higher saving rate reaches retirement sooner.",
        "source": "Fidelity: save more than 15% to retire before 67. Bengen (1994), the 4% rule: aim for 25x yearly spending.",
    },
)
PERCENT_WHOLE = 100

WARN_AT = 0.9  # share of a limit at which the meter turns to "near limit"
MAX_AMOUNT_CENTS = 10_000_000 * 100  # $10M; anything larger is a typo
MAX_NAME_LENGTH = 40  # characters
CENTS_PER_DOLLAR = 100


def empty_budget():
    """A new user's budget: no limit, no goal, no months."""
    return {
        "settings": {"savings_goal": 0, "limits": {"overall": 0}, "targets": {}},
        "months": {},
    }


def parse_amount(text):
    """Turn what the user typed into cents, or raise ValueError saying what is wrong."""
    cleaned = str(text if text is not None else "").strip().replace("$", "").replace(",", "")
    if not cleaned:
        raise ValueError("Enter an amount")
    if cleaned.startswith("-"):
        raise ValueError("Amount can't be negative")
    whole, _, fraction = cleaned.partition(".")
    if not (whole or fraction) or not (whole or "0").isdigit() or (fraction and not fraction.isdigit()):
        raise ValueError("Amount must be a number, like 12.50")
    if len(fraction) > 2:
        raise ValueError("Amount can have at most 2 decimal places")
    cents = int(whole or 0) * CENTS_PER_DOLLAR + int(fraction.ljust(2, "0"))
    if cents == 0:
        raise ValueError("Amount must be more than $0")
    if cents > MAX_AMOUNT_CENTS:
        raise ValueError("Amount is too large")
    return cents


def parse_optional_amount(text):
    """Like parse_amount, but blank or 0 means "not set" and returns 0."""
    cleaned = str(text if text is not None else "").strip().replace("$", "").replace(",", "")
    if cleaned.strip("0.") == "":
        return 0
    return parse_amount(cleaned)


def clean_name(text):
    """Trim a line item's name, or raise ValueError if it is unusable."""
    name = " ".join(str(text if text is not None else "").split())
    if not name:
        raise ValueError("Enter a name")
    if len(name) > MAX_NAME_LENGTH:
        raise ValueError(f"Name must be {MAX_NAME_LENGTH} characters or fewer")
    return name


def clean_tier(text):
    """Trim an investment tier's name, or raise ValueError if it is unusable."""
    tier = " ".join(str(text if text is not None else "").split())
    if not tier:
        raise ValueError("Enter a tier, like Retirement")
    if len(tier) > MAX_NAME_LENGTH:
        raise ValueError(f"Tier must be {MAX_NAME_LENGTH} characters or fewer")
    return tier


def make_item(section, name_text, amount_text, tier_text=None):
    """A validated line. Investment lines also carry their tier."""
    item = {"name": clean_name(name_text), "amount": parse_amount(amount_text)}
    if section == TIERED_SECTION:
        item["tier"] = clean_tier(tier_text)
    return item


def format_money(cents):
    """$1,200 for whole dollars, $12.50 otherwise, -$85 when negative."""
    sign = "-" if cents < 0 else ""
    dollars, remainder = divmod(abs(cents), CENTS_PER_DOLLAR)
    if remainder:
        return f"{sign}${dollars:,}.{remainder:02d}"
    return f"{sign}${dollars:,}"


# ---------- Subcategories ----------

def missing_presets(budget, subs):
    """Presets a saved list lacks (saved before subcategories, or before a preset was added), unless removed."""
    keys = {sub["key"] for sub in subs} | set(budget["settings"].get("removed_presets", []))
    return [dict(preset) for preset in PRESET_SUBCATEGORIES if preset["key"] not in keys]


def subcategories(budget):
    """Every subcategory in page order within its category."""
    stored = budget["settings"].get("subcategories", [])
    return stored + missing_presets(budget, stored)


def editable_subcategories(budget):
    """The stored list to change, with any missing presets written in."""
    subs = budget["settings"].setdefault("subcategories", [])
    subs.extend(missing_presets(budget, subs))
    return subs


def subcategories_in(budget, category):
    return [sub for sub in subcategories(budget) if sub["category"] == category]


def section_keys(budget):
    """Income first, then every subcategory key: what a month holds lines under."""
    return [INCOME] + [sub["key"] for sub in subcategories(budget)]


def find_subcategory(budget, key):
    for sub in editable_subcategories(budget):
        if sub["key"] == key:
            return sub
    raise ValueError(f"Unknown subcategory: {key}")


def check_category(category):
    if category not in CATEGORIES:
        raise ValueError("Pick a category: Needs, Wants, Investing or Savings")


def unused_name(budget, name_text, except_key=None):
    """A cleaned subcategory name nobody else has, ignoring capitals."""
    name = clean_name(name_text)
    # History zooms by clicking a name, so a subcategory can't share one with a category or the unzoomed view.
    reserved = {INCOME, ALL_SPENDING.lower()} | {category_name.lower() for category_name in CATEGORY_NAMES.values()}
    taken = {sub["name"].lower() for sub in subcategories(budget) if sub["key"] != except_key} | reserved
    if name.lower() in taken:
        raise ValueError(f"You already have {name}")
    return name


def add_subcategory(budget, name_text, category):
    """A new subcategory of the user's own, at the end of its category; returns it."""
    check_category(category)
    name = unused_name(budget, name_text)
    subs = editable_subcategories(budget)
    if sum(sub["key"] not in PRESET_KEYS for sub in subs) >= MAX_CUSTOM_SUBCATEGORIES:
        raise ValueError(f"You can add up to {MAX_CUSTOM_SUBCATEGORIES} of your own subcategories")
    # The key is fixed for good, so a rename never detaches past months' lines.
    base = UNSAFE_KEY_CHARACTERS.sub("_", name.lower()).strip("_") or "subcategory"
    taken = set(section_keys(budget))
    key, number = base, 2
    while key in taken:
        key, number = f"{base}_{number}", number + 1
    sub = {"key": key, "name": name, "category": category}
    subs.append(sub)
    return sub


def rename_subcategory(budget, key, name_text):
    find_subcategory(budget, key)["name"] = unused_name(budget, name_text, except_key=key)


def move_subcategory(budget, key, category):
    """Put a subcategory under another category. It goes to the end of that category's list."""
    check_category(category)
    subs = editable_subcategories(budget)
    sub = find_subcategory(budget, key)
    if sub["category"] != category:
        subs.remove(sub)
        subs.append({**sub, "category": category})


def remove_subcategory(budget, key):
    """Delete a subcategory, preset or not. Refused while any month has lines in it, so nothing is lost."""
    sub = find_subcategory(budget, key)
    used = sum(1 for month in budget["months"].values() if month.get(key))
    if used:
        plural = "s" if used > 1 else ""
        raise ValueError(f"{sub['name']} has items in {used} month{plural}. Delete them first.")
    editable_subcategories(budget).remove(sub)
    if key in PRESET_KEYS:  # so it isn't added back as "missing"
        budget["settings"].setdefault("removed_presets", []).append(key)
    for month in budget["months"].values():
        month.pop(key, None)


# ---------- Totals ----------

def section_total(month, section):
    """Sum of every line in one section."""
    return sum(item["amount"] for item in month.get(section, []))


def category_total(budget, month, category):
    """Sum of every subcategory in one category."""
    return sum(section_total(month, sub["key"]) for sub in subcategories_in(budget, category))


def total_expenses(budget, month):
    """Needs + wants: what limits and "spent" cover."""
    return sum(category_total(budget, month, category) for category in SPENDING_CATEGORIES)


def total_invested(budget, month):
    """Everything put into investing subcategories this month."""
    return category_total(budget, month, "investing")


def total_set_aside(budget, month):
    """Money moved into savings subcategories on purpose, like an emergency fund."""
    return category_total(budget, month, "savings")


def leftover(budget, month):
    """Income minus everything that went out. Negative means over budget."""
    return section_total(month, INCOME) - sum(category_total(budget, month, category) for category in CATEGORIES)


def saved(budget, month):
    """Set aside plus whatever is left over (never below the set-aside)."""
    return total_set_aside(budget, month) + max(leftover(budget, month), 0)


def meter(spent, limit):
    """State of a spending limit: none (no limit set), good, warn or over, with its label."""
    if limit <= 0:
        return {"state": "none", "fraction": 0.0, "label": ""}
    fraction = spent / limit
    if spent > limit:
        return {"state": "over", "fraction": 1.0, "label": f"Over by {format_money(spent - limit)}"}
    if fraction >= WARN_AT:
        return {"state": "warn", "fraction": fraction, "label": f"Near limit, {format_money(limit - spent)} left"}
    return {"state": "good", "fraction": fraction, "label": f"{format_money(limit - spent)} left"}


def savings_progress(budget):
    """Saved so far, every month added up."""
    return sum(saved(budget, get_month(budget, key)) for key in budget["months"])


# ---------- Lines ----------

def editable_month(budget, month_key):
    """The stored month to change, created if new, with every section present (older saves lack some)."""
    month = budget["months"].setdefault(month_key, {})
    for section in section_keys(budget):
        month.setdefault(section, [])
    return month


def same_tier_as_before(budget, item):
    """Typing "retirement" files the line under an existing "Retirement" tier rather than a new one."""
    if "tier" in item:
        existing = {tier.lower(): tier for tier in tiers(budget)}
        item["tier"] = existing.get(item["tier"].lower(), item["tier"])
    return item


def add_item(budget, month_key, section, name_text, amount_text, tier_text=None):
    """Validate and add a line to a section; returns the added item."""
    if section not in section_keys(budget):
        raise ValueError(f"Unknown section: {section}")
    item = same_tier_as_before(budget, make_item(section, name_text, amount_text, tier_text))
    editable_month(budget, month_key)[section].append(item)
    return item


def remove_item(budget, month_key, section, index):
    """Delete one line from a section by its position."""
    items = editable_month(budget, month_key)[section]
    if not 0 <= index < len(items):
        raise IndexError(f"No line {index} in {section}")
    items.pop(index)


def update_item(budget, month_key, section, index, name_text, amount_text, tier_text=None):
    """Validate and replace one line; returns the new item."""
    items = editable_month(budget, month_key)[section]
    if not 0 <= index < len(items):
        raise IndexError(f"No line {index} in {section}")
    item = same_tier_as_before(budget, make_item(section, name_text, amount_text, tier_text))
    items[index] = item
    return item


def tiers(budget):
    """Every investment tier used in any month, in the order they were first used."""
    seen = {}
    for key in sorted(budget["months"]):
        for item in get_month(budget, key).get(TIERED_SECTION, []):
            seen.setdefault(item["tier"], None)
    return list(seen)


# ---------- Percentage targets ----------

def target_number(text):
    """A typed percentage as a whole number from 0 to 100, or None if it isn't one."""
    cleaned = str(text if text is not None else "").strip().rstrip("%").strip()
    return int(cleaned) if cleaned.isdigit() and int(cleaned) <= PERCENT_WHOLE else None


def is_blank(text):
    return not str(text if text is not None else "").strip()


def parse_targets(needs_text, wants_text, investing_text, savings_text):
    """Turn four typed percentages into targets. All blank means targets are off ({})."""
    typed = dict(zip(CATEGORIES, (needs_text, wants_text, investing_text, savings_text)))
    if all(is_blank(text) for text in typed.values()):
        return {}
    targets = {}
    for key, text in typed.items():
        targets[key] = target_number(text)
        if targets[key] is None:
            raise ValueError(f"{CATEGORY_NAMES[key]} must be a whole number from 0 to 100")
    total = sum(targets.values())
    if total != PERCENT_WHOLE:
        raise ValueError(f"Targets must add up to 100% (now {total}%)")
    return targets


def complete_targets(texts, auto_key, edited_key):
    """Fill the one blank target box with what makes 100, and keep it in step while the other three change.

    Returns the four box texts and the box filled in for the user (None once they type in it themselves). The box
    that changed is never rewritten, so the caller only needs to update the other three.
    """
    values = dict(zip(CATEGORIES, texts))

    def remainder(key):
        others = [target_number(text) for other, text in values.items() if other != key]
        return str(PERCENT_WHOLE - sum(others)) if None not in others and sum(others) <= PERCENT_WHOLE else ""

    if edited_key == auto_key:
        # The filled-in box changed: either this function's own update, or the user typing over it.
        return list(texts), auto_key if values[auto_key] == remainder(auto_key) else None
    if auto_key is None:
        blanks = [key for key, text in values.items() if is_blank(text)]
        # Never fill the box being typed in: clearing a box means the user is about to type a new number there.
        if len(blanks) != 1 or edited_key in blanks or not remainder(blanks[0]):
            return list(texts), None
        auto_key = blanks[0]
    values[auto_key] = remainder(auto_key)
    return [values[key] for key in CATEGORIES], auto_key


def share_of(cents, percent):
    """percent of an amount, rounded down to the cent."""
    return cents * percent // PERCENT_WHOLE


def plan_amounts(limit_text, percent_texts):
    """While typing on Goals: each category's dollars for every valid %. Empty until a limit is typed."""
    try:
        limit = parse_optional_amount(limit_text)
    except ValueError:
        return {}
    percents = {category: target_number(text) for category, text in zip(CATEGORIES, percent_texts)}
    return {category: share_of(limit, percent) for category, percent in percents.items() if limit and percent is not None}


def category_limits(settings):
    """Each category's dollar share of the monthly limit by its target %. Empty unless both are set."""
    overall = settings["limits"].get("overall", 0)
    targets = settings.get("targets", {})
    if overall <= 0 or set(targets) != set(CATEGORIES):
        return {}
    return {category: share_of(overall, targets[category]) for category in CATEGORIES}


def minimum_meter(reached, goal):
    """A target to reach rather than stay under: short is amber, reaching it is good."""
    if goal <= 0:
        return {"state": "none", "fraction": 0.0, "label": ""}
    if reached >= goal:
        return {"state": "good", "fraction": 1.0, "label": "On target"}
    return {"state": "warn", "fraction": reached / goal, "label": f"{format_money(goal - reached)} short"}


def target_meters(budget, targets, month):
    """Each category's meter, with the actual and target percent of income. Empty when off or no income."""
    income = section_total(month, INCOME)
    # Targets saved before investing had its own share have three keys; they count as off.
    if set(targets) != set(CATEGORIES) or income <= 0:
        return {}
    reached = {category: category_total(budget, month, category) for category in CATEGORIES}
    reached["savings"] = saved(budget, month)
    meters = {}
    for category in CATEGORIES:
        goal = income * targets[category] // PERCENT_WHOLE
        check = meter if category in SPENDING_CATEGORIES else minimum_meter
        meters[category] = {
            **check(reached[category], goal),
            "actual_percent": round(reached[category] * PERCENT_WHOLE / income),
            "target_percent": targets[category],
        }
    return meters


def get_month(budget, month_key):
    """The month's data with every section present, or an empty month if nothing was entered yet."""
    return {**{section: [] for section in section_keys(budget)}, **budget["months"].get(month_key, {})}


def month_key(day=None):
    """'2026-10' for any date in October 2026; today by default."""
    day = day or date.today()
    return f"{day.year}-{day.month:02d}"


def shift_month(key, steps):
    """Move a month key forward or back: shift_month('2026-12', 1) is '2027-01'."""
    year, month = (int(part) for part in key.split("-"))
    index = year * 12 + (month - 1) + steps
    return f"{index // 12}-{index % 12 + 1:02d}"


def month_name(key):
    """'October' for '2026-10'."""
    year, month = (int(part) for part in key.split("-"))
    return date(year, month, 1).strftime("%B")


def month_label(key):
    """'October 2026' for '2026-10'."""
    return f"{month_name(key)} {key.split('-')[0]}"


# ---------- History ----------

HISTORY_RANGES = {"6 months": 6, "12 months": 12}  # "All" is the third choice, from the first month with data
MAX_HISTORY_MONTHS = 24  # columns stay readable up to two years
SAMPLE_TAG = " (sample)"

# Five believable earlier months for demos, oldest first: (section, name, dollars), plus a tier for investments.
SAMPLE_ROTH = ("investments", "Roth IRA", 250, "Retirement")
SAMPLE_INDEX = ("investments", "Index fund", 150, "Brokerage")
SAMPLE_MONTHS = [
    [("income", "Salary", 3000), ("necessities", "Rent", 1200), ("necessities", "Groceries", 340),
     ("necessities", "Gas", 110), ("discretionary", "Streaming", 28), ("discretionary", "Dinner out", 90),
     SAMPLE_ROTH],
    [("income", "Salary", 3000), ("income", "Freelance", 250), ("necessities", "Rent", 1200),
     ("necessities", "Groceries", 365), ("necessities", "Gas", 95), ("discretionary", "Concert tickets", 140),
     ("discretionary", "Streaming", 28), SAMPLE_ROTH, SAMPLE_INDEX],
    [("income", "Salary", 3000), ("necessities", "Rent", 1200), ("necessities", "Groceries", 410),
     ("necessities", "Gas", 120), ("discretionary", "Streaming", 28), ("travel", "Flights", 380),
     ("travel", "Hotel", 420), ("travel", "Food on the trip", 160), SAMPLE_ROTH],
    [("income", "Salary", 3000), ("income", "Freelance", 400), ("necessities", "Rent", 1200),
     ("necessities", "Groceries", 350), ("necessities", "Gas", 105), ("discretionary", "Video game", 70),
     ("discretionary", "Streaming", 28), SAMPLE_ROTH, SAMPLE_INDEX],
    [("income", "Salary", 3000), ("necessities", "Rent", 1200), ("necessities", "Groceries", 375),
     ("necessities", "Gas", 115), ("discretionary", "Movies", 45), ("discretionary", "Streaming", 28),
     ("travel", "Weekend train", 90), SAMPLE_ROTH, SAMPLE_INDEX],
]
def month_history(budget, end_key, count):
    """One row per month, oldest first, ending at end_key. Months with nothing entered are zeros."""
    rows = []
    for steps in range(count - 1, -1, -1):
        key = shift_month(end_key, -steps)
        month = get_month(budget, key)
        left = leftover(budget, month)
        rows.append({
            "key": key,
            "label": month_name(key)[:3],
            "income": section_total(month, INCOME),
            "spent": total_expenses(budget, month),
            "invested": total_invested(budget, month),
            "left": left,
            "saved": total_set_aside(budget, month) + left,  # negative when over budget
            "categories": {category: category_total(budget, month, category) for category in CATEGORIES},
            "subcategories": {sub["key"]: section_total(month, sub["key"]) for sub in subcategories(budget)},
        })
    return rows


def history_range(budget, choice, today_key):
    """The (end_key, count) a range choice covers. 'All' runs from the first month with data to today."""
    if choice in HISTORY_RANGES:
        return today_key, HISTORY_RANGES[choice]
    keys = [key for key, month in budget["months"].items() if any(month.values()) and key <= today_key]
    if not keys:
        return today_key, 1
    first = min(keys)
    first_year, first_month = (int(part) for part in first.split("-"))
    year, month = (int(part) for part in today_key.split("-"))
    span = (year - first_year) * 12 + (month - first_month) + 1
    return today_key, min(span, MAX_HISTORY_MONTHS)


def months_with_data(budget):
    """How many months have at least one line."""
    return sum(1 for month in budget["months"].values() if any(month.values()))


def line_label(item):
    """How a line is named in History: investment lines carry their tier, "Retirement: Index fund"."""
    return f"{item['tier']}: {item['name']}" if "tier" in item else item["name"]


def top_items(budget, keys, section, limit=5):
    """The biggest lines in a section across the given months: (name, month_key, cents), largest first."""
    items = [(line_label(item), key, item["amount"]) for key in keys for item in get_month(budget, key)[section]]
    return sorted(items, key=lambda item: item[2], reverse=True)[:limit]


def section_lines(budget, keys, section):
    """Each month's lines in one section as {month_key: {label: cents}}; lines sharing a label add up."""
    months = {}
    for key in keys:
        totals = months.setdefault(key, {})
        for item in get_month(budget, key)[section]:
            label = line_label(item)
            totals[label] = totals.get(label, 0) + item["amount"]
    return months


def fill_sample_months(budget, current_key):
    """Fill the five months before current_key with sample lines, skipping any that already have data."""
    filled = 0
    for steps, lines in zip(range(len(SAMPLE_MONTHS), 0, -1), SAMPLE_MONTHS):
        key = shift_month(current_key, -steps)
        if any(get_month(budget, key).values()):
            continue
        for section, name, dollars, *tier in lines:
            if section not in section_keys(budget):  # a preset the user removed
                continue
            add_item(budget, key, section, name + SAMPLE_TAG, str(dollars), *tier)
        filled += 1
    return filled


## `storage.py`

Saves each name's budget as a JSON file, with a salted hash of its password.

In [ ]:
%%writefile storage.py
"""Saved budgets for Sliced Bread: one JSON file per name in DATA_DIR, with a salted password hash inside."""

import hashlib
import hmac
import json
import os
import re
import secrets

import budget

# Set SLICED_BREAD_DATA to save budgets somewhere other than ./data.
DATA_DIR = os.environ.get("SLICED_BREAD_DATA", "data")
MAX_NAME_LENGTH = 30  # characters
UNSAFE_FILE_CHARACTERS = re.compile(r"[^a-z0-9_-]+")
MIN_PASSWORD_LENGTH = 8  # characters
PASSWORD_ITERATIONS = 600_000  # PBKDF2-SHA256 rounds, OWASP's 2023 figure
SALT_BYTES = 16
MAX_EMAIL_LENGTH = 254  # characters, the longest an address can be
EMAIL_SHAPE = re.compile(r"[^@\s]+@[^@\s]+\.[^@\s]+")


def clean_name(text):
    """The name as it will be shown: trimmed, single spaces. Raises ValueError if unusable."""
    name = " ".join((text or "").split())
    if not name:
        raise ValueError("Enter your name")
    if len(name) > MAX_NAME_LENGTH:
        raise ValueError(f"Name must be {MAX_NAME_LENGTH} characters or fewer")
    return name


def file_key(name):
    """The file a name's budget lives in: 'Mark G' is 'mark_g'. Anything that isn't a letter or digit is dropped."""
    key = UNSAFE_FILE_CHARACTERS.sub("_", clean_name(name).lower()).strip("_")
    if not key:
        raise ValueError("Name needs at least one letter or number")
    return key


def _path(key):
    return os.path.join(DATA_DIR, f"{key}.json")


def load_budget(key):
    """The saved budget for this key, or a new empty one."""
    try:
        with open(_path(key), encoding="utf-8") as file:
            return json.load(file)
    except FileNotFoundError:
        return budget.empty_budget()


def save_budget(key, user_budget):
    """Write to a temp file first, then swap it in, so a crash never leaves half a file."""
    os.makedirs(DATA_DIR, exist_ok=True)
    temp_path = _path(key) + ".tmp"
    with open(temp_path, "w", encoding="utf-8") as file:
        json.dump(user_budget, file, indent=2)
    os.replace(temp_path, _path(key))


def delete_budget(key):
    """Remove this name's saved budget for good."""
    if os.path.exists(_path(key)):
        os.remove(_path(key))


def rename_budget(old_key, new_key):
    """Move a budget to a new name's file. Refuses a name someone else already has."""
    if os.path.exists(_path(new_key)):
        raise ValueError("That name is taken")
    os.replace(_path(old_key), _path(new_key))


# ---------- Passwords ----------

def hash_password(password, salt_hex=None):
    """The salt and PBKDF2 hash to store for a password, both as hex. A new salt unless one is given."""
    salt = bytes.fromhex(salt_hex) if salt_hex else secrets.token_bytes(SALT_BYTES)
    digest = hashlib.pbkdf2_hmac("sha256", password.encode("utf-8"), salt, PASSWORD_ITERATIONS)
    return {"salt": salt.hex(), "password_hash": digest.hex()}


def check_password(account, password):
    """Whether password matches the account's stored hash, compared in constant time."""
    expected = hash_password(password or "", account["salt"])["password_hash"]
    return hmac.compare_digest(expected, account["password_hash"])


def new_password(password):
    """Hash a password being set, or raise ValueError if it is too short."""
    if len(password or "") < MIN_PASSWORD_LENGTH:
        raise ValueError(f"Password must be at least {MIN_PASSWORD_LENGTH} characters")
    return hash_password(password)


def clean_email(text):
    """A trimmed email address, blank allowed. Raises ValueError if it can't be one."""
    email = (text or "").strip()
    if email and (len(email) > MAX_EMAIL_LENGTH or not EMAIL_SHAPE.fullmatch(email)):
        raise ValueError("Email should look like name@example.com")
    return email


def sign_in(name_text, password):
    """(file key, budget) for a name and password. A new name, or a budget saved before passwords, takes this one."""
    name = clean_name(name_text)
    key = file_key(name)
    user_budget = load_budget(key)
    account = user_budget.get("account", {})
    if account.get("password_hash"):
        if not check_password(account, password):
            raise ValueError("That name and password don't match")
        return key, user_budget
    user_budget["account"] = {"name": name, "email": "", **new_password(password)}
    save_budget(key, user_budget)
    return key, user_budget


## `design.py`

Pond, the design system: colours, type, spacing, the Gradio theme, the CSS and the mallard mascot.

In [ ]:
%%writefile design.py
"""Pond, the Sliced Bread design system: tokens, the Gradio theme, the CSS and the mallard. See PLAN.md section 5.

The one rule: green is the app's colour. It marks what you can press and spending that is on track.
Amber means near a limit, red means over. Nothing in app.py sets a colour or size by hand.
"""

import html
import math

import gradio as gr

import budget

# ---------- Tokens ----------

PAPER = "#F4F2ED"  # page background, a shade darker than cards so they stand off it
PAPER_DEEP = "#E9E5DD"  # the tab switcher's track
CARD = "#FFFFFF"  # cards
INK = "#1C1B18"  # all text, all money
MUTED = "#6B665D"  # labels, 5:1 on paper
LINE = "#E2DDD3"  # card borders, dividers, empty meter track
FIELD_BORDER = "#958D81"  # text box borders, 3.3:1 on white so a box reads as a box
ACCENT = "#1F7A4D"  # mallard green: buttons, focus, on-track meters; white text on it is 5.3:1
ACCENT_DARK = "#17603C"  # button hover
ACCENT_SOFT = "#E7F1EA"  # add/edit panel, Investments card wash
HOVER = "#F0EDE7"  # neutral hover for quiet icon buttons
WARN = "#D99A00"  # meter 90-100% (fill only, never text)
OVER = "#B42318"  # meter over 100%, "Over budget" text

# Fills only (dots, bars, chart columns), never text. Checked with the dataviz palette validator:
# all pairs pass lightness, chroma, colour-blind and normal-vision separation on white (2026-11-03).
INCOME_COLOR = ACCENT
CATEGORY_COLORS = {"needs": "#2A78D6", "wants": "#EB6834", "investing": "#6250D6", "savings": "#1BAF7A"}
# A category's subcategories run from a little darker than its colour to a little lighter, so the dot says the category.
SUBCATEGORY_DARKEST_MIX = 0.2  # share of black in the first subcategory's shade (when there is more than one)
SUBCATEGORY_LIGHTEST_MIX = 0.35  # share of white in the last
# History zoom: a category's lines run from its colour mixed toward black (biggest) to toward white (smallest).
DARKEST_MIX = 0.35  # share of black in the biggest line's shade
LIGHTEST_MIX = 0.6  # share of white in the smallest; any lighter fades into the white card
ITEM_LIGHTEST_MIX = 0.4  # same, from a subcategory's colour, which may already be up to 35% white
METER_COLORS = {"good": ACCENT, "warn": WARN, "over": OVER}
# A spending limit's fill blends through these as it fills: (share of the limit spent, colour). Red at the limit.
LIMIT_COLOR_STOPS = ((0.0, ACCENT), (budget.WARN_AT, WARN), (1.0, OVER))

# The mallard's own colours. Duck only; never used for UI.
DUCK_HEAD = "#1D6B47"
DUCK_BILL = "#E8B931"
DUCK_BREAST = "#7A3E1D"
DUCK_BODY = "#B9B3A8"

# The hero donut, in px.
DONUT_SIZE = 148
DONUT_STROKE = 18  # ring thickness: thin, so the number in the middle leads
DONUT_GAP = 2  # card-coloured gap between segments

FONT_TEXT = "Geist"
FONT_DISPLAY = "Instrument Serif"
PAGE_MAX_WIDTH = "640px"

# Amount boxes (class sb-money-input) show "$1,234.56" as you type. The caret keeps its place among the digits, and
# a fresh input event tells Gradio the new text; formatting twice changes nothing, so that event stops there.
MONEY_INPUT_SCRIPT = r"""<script>
function sbMoneyText(text) {
  var cleaned = text.replace(/[^0-9.]/g, '');
  if (!cleaned) return '';
  var point = cleaned.indexOf('.');
  var whole = (point < 0 ? cleaned : cleaned.slice(0, point)).replace(/^0+(?=\d)/, '');
  var fraction = point < 0 ? null : cleaned.slice(point + 1).replace(/\./g, '').slice(0, 2);
  whole = whole.replace(/\B(?=(\d{3})+(?!\d))/g, ',') || (fraction !== null ? '0' : '');
  return '$' + whole + (fraction !== null ? '.' + fraction : '');
}
document.addEventListener('input', function (event) {
  var box = event.target;
  if (!box.closest || !box.closest('.sb-money-input')) return;
  var typed = box.value, formatted = sbMoneyText(typed);
  if (formatted === typed) return;
  var after = typed.slice(box.selectionEnd || typed.length).replace(/[^0-9.]/g, '').length;
  var caret = formatted.length, seen = 0;
  while (caret > 0 && seen < after) { caret--; if (/[0-9.]/.test(formatted[caret])) seen++; }
  box.value = formatted;
  box.setSelectionRange(caret, caret);
  box.dispatchEvent(new Event('input', {bubbles: true}));
});
</script>"""

# Loaded in <head> so both faces are ready before first paint.
HEAD = (
    '<link rel="preconnect" href="https://fonts.googleapis.com">'
    '<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>'
    '<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Geist:wght@400;500;600&'
    'family=Instrument+Serif:ital@0;1&display=swap">'
    # History: clicking anything with data-sb-zoom picks that category in the hidden radio, which redraws the chart.
    "<script>document.addEventListener('click', function (event) {"
    "  var target = event.target.closest && event.target.closest('[data-sb-zoom]');"
    "  if (!target) return;"
    "  var choice = target.getAttribute('data-sb-zoom');"
    "  document.querySelectorAll('.sb-category-pick label').forEach(function (label) {"
    "    if (label.textContent.trim() === choice) label.querySelector('input').click();"
    "  });"
    "});</script>"
    + MONEY_INPUT_SCRIPT
)

# Green and paper as Gradio colour ramps, so built-in components pick them up.
ACCENT_RAMP = gr.themes.Color(
    c50="#F2F8F4", c100=ACCENT_SOFT, c200="#C5DFCF", c300="#94C4A6", c400="#5BA37A", c500="#2F8A5B",
    c600=ACCENT, c700=ACCENT_DARK, c800="#124A2F", c900="#0D3622", c950="#082416", name="accent",
)
PAPER_RAMP = gr.themes.Color(
    c50=PAPER, c100=PAPER_DEEP, c200=LINE, c300="#C9C1B4", c400=FIELD_BORDER, c500="#7D776D",
    c600=MUTED, c700="#4A4640", c800="#2F2C28", c900=INK, c950="#141311", name="paper",
)


def build_theme():
    """The Gradio theme. Light only: every dark-mode value is copied from its light twin."""
    theme = gr.themes.Base(
        primary_hue=ACCENT_RAMP,
        secondary_hue=ACCENT_RAMP,
        neutral_hue=PAPER_RAMP,
        font=[FONT_TEXT, "ui-sans-serif", "system-ui", "sans-serif"],
        font_mono=["ui-monospace", "Consolas", "monospace"],
        text_size=gr.themes.sizes.text_md,
    ).set(
        body_background_fill=PAPER,
        body_text_color=INK,
        body_text_color_subdued=MUTED,
        background_fill_primary=CARD,
        background_fill_secondary=PAPER,
        border_color_primary=LINE,
        border_color_accent=LINE,
        color_accent=ACCENT,
        color_accent_soft=ACCENT_SOFT,
        block_background_fill=CARD,
        block_border_color=LINE,
        block_border_width="0px",
        block_shadow="none",
        block_padding="0px",
        block_title_text_color=MUTED,
        block_title_text_size="14px",
        block_title_text_weight="500",
        block_label_text_color=MUTED,
        input_background_fill=CARD,
        input_border_width="1px",
        input_border_color=FIELD_BORDER,
        input_border_color_hover=INK,
        input_border_color_focus=ACCENT,
        input_shadow="none",
        input_shadow_focus="none",
        input_radius="12px",
        input_padding="12px 14px",
        input_text_size="16px",
        button_large_radius="999px",
        button_small_radius="999px",
        button_large_padding="12px 22px",
        button_large_text_size="16px",
        button_large_text_weight="600",
        button_border_width="1px",
        **{f"button_{kind}_shadow{state}": "none" for kind in ("primary", "secondary", "cancel") for state in ("", "_hover", "_active")},
        layout_gap="16px",
        button_primary_background_fill=ACCENT,
        button_primary_background_fill_hover=ACCENT_DARK,
        button_primary_text_color="#FFFFFF",
        button_primary_border_color=ACCENT,
        button_primary_border_color_hover=ACCENT_DARK,
        button_secondary_background_fill=CARD,
        button_secondary_background_fill_hover=HOVER,
        button_secondary_text_color=INK,
        button_secondary_border_color=LINE,
        button_secondary_border_color_hover=FIELD_BORDER,
        button_cancel_background_fill=CARD,
        button_cancel_background_fill_hover="#FBEDEB",
        button_cancel_text_color=OVER,
        button_cancel_border_color="#E8B4AE",
    )
    for name in list(vars(theme)):
        if name.endswith("_dark"):
            setattr(theme, name, getattr(theme, name.removesuffix("_dark")))
    return theme


CSS = f"""
:root {{
  --sb-paper: {PAPER}; --sb-paper-deep: {PAPER_DEEP}; --sb-card: {CARD}; --sb-ink: {INK}; --sb-muted: {MUTED};
  --sb-line: {LINE}; --sb-accent: {ACCENT}; --sb-accent-dark: {ACCENT_DARK}; --sb-accent-soft: {ACCENT_SOFT};
  --sb-hover: {HOVER}; --sb-over: {OVER};
  --sb-serif: '{FONT_DISPLAY}', Georgia, 'Times New Roman', serif;
  --sb-sans: '{FONT_TEXT}', ui-sans-serif, system-ui, sans-serif;
}}

/* One centred column, phone first. */
.gradio-container {{
  width: 100% !important; max-width: {PAGE_MAX_WIDTH} !important; margin: 0 auto !important; padding: 20px 16px 48px !important;
  font-family: var(--sb-sans) !important; -webkit-font-smoothing: antialiased;
}}
.gradio-container .main {{ padding: 0 !important; }}
.gradio-container .main > .wrap > .contain > .column {{ gap: 16px !important; }}
footer {{ display: none !important; }}

/* No backlights: Gradio's busy fade, accent border and progress overlay are off; keyboard focus keeps a thin ring. */
.pending {{ opacity: 1 !important; animation: none !important; }}
.block.border_focus, .block.border_contrast {{ border-color: transparent !important; }}
.wrap.default, .wrap.center, .wrap.translucent, .eta-bar, .progress-level, .progress-text, .meta-text {{ display: none !important; }}
:focus {{ outline: none; }}
:focus-visible {{ outline: 2px solid var(--sb-accent) !important; outline-offset: 2px; box-shadow: none !important; }}
textarea:focus-visible, input:focus-visible {{ outline: none !important; }}

/* Type: 14 / 16 / 20 / 26 / 40 / 64. Serif for display, Geist for everything else. Nothing under 14. */
.sb-serif, .sb-wordmark, .sb-title, .sb-hero-amount, .sb-big {{
  font-family: var(--sb-serif); font-weight: 400; letter-spacing: -0.01em;
}}
.sb-wordmark {{ font-size: 26px; line-height: 1; }}
.sb-title {{ font-size: 26px; line-height: 1.1; display: flex; align-items: center; gap: 10px; }}
.sb-eyebrow {{ font-size: 14px; font-weight: 500; color: var(--sb-muted); }}
.sb-note {{ font-size: 14px; color: var(--sb-muted); line-height: 1.45; }}
.sb-money {{ font-variant-numeric: tabular-nums; white-space: nowrap; font-weight: 500; }}
.sb-hero-amount {{ font-size: 64px; line-height: 1; margin-top: 6px; font-variant-numeric: tabular-nums; }}
.sb-hero-amount.sb-over {{ color: var(--sb-over); }}
.sb-big {{ font-size: 40px; line-height: 1; font-variant-numeric: tabular-nums; }}
@media (max-width: 480px) {{
  .sb-hero-amount {{ font-size: 48px; }}
  .sb-title {{ font-size: 24px; }}
}}
.sb-dot {{ width: 10px; height: 10px; border-radius: 999px; display: inline-block; flex: none; }}
.sb-marker {{
  display: inline-flex; align-items: center; gap: 6px; padding: 2px 10px 2px 8px; border-radius: 999px; flex: none;
  border: 1px solid var(--sb-line); background: var(--sb-card); font-family: var(--sb-sans); font-size: 14px;
  font-weight: 500; line-height: 1.4; color: var(--sb-muted); letter-spacing: 0; white-space: nowrap;
}}
.sb-marker .sb-dot {{ width: 8px; height: 8px; }}
.html-container, .prose {{ padding: 0 !important; }}

/* Cards: white on paper, 20px corners, one soft shadow. */
.sb-card {{
  background: var(--sb-card) !important; border: 1px solid var(--sb-line) !important; border-radius: 20px !important;
  padding: 20px !important; gap: 14px !important; box-shadow: 0 1px 2px rgb(28 27 24 / 0.04), 0 4px 16px rgb(28 27 24 / 0.04);
}}
@media (min-width: 600px) {{ .sb-card {{ padding: 24px !important; }} }}
.sb-card-accent {{ background: var(--sb-accent-soft) !important; border-color: #CFE3D6 !important; }}
.sb-row {{ align-items: center !important; gap: 8px !important; flex-wrap: nowrap !important; }}
.sb-grow {{ flex: 1 1 auto !important; min-width: 0 !important; }}
.sb-head {{ display: flex; justify-content: space-between; align-items: center; gap: 12px; }}
.sb-head-total {{ display: flex; flex-direction: column; align-items: flex-end; gap: 2px; }}
.sb-head .sb-money {{ font-size: 16px; }}
.sb-chip {{
  display: inline-flex; align-items: center; gap: 6px; padding: 6px 12px; border-radius: 999px;
  background: var(--sb-accent-soft); color: var(--sb-accent-dark); font-size: 14px; font-weight: 500;
}}

/* Header: logo and wordmark left; the person's name and an avatar button right, which opens a small menu. */
.sb-app {{ position: relative; }}
.sb-header {{ align-items: center !important; flex-wrap: nowrap !important; gap: 10px !important; padding: 4px 4px 0; }}
.sb-brand {{ display: flex; align-items: center; gap: 10px; }}
.sb-person {{ flex: none !important; width: auto !important; }}
.sb-person-name {{ font-size: 14px; font-weight: 500; white-space: nowrap; }}
@media (max-width: 400px) {{ .sb-person-name {{ display: none; }} }}
.sb-avatar-button {{
  width: 40px !important; height: 40px; min-width: 40px !important; max-width: 40px; padding: 0 !important; flex: none !important;
  border: none !important; border-radius: 999px !important; background: var(--sb-accent-soft) !important;
  color: var(--sb-accent-dark) !important; font-weight: 600 !important; font-size: 16px !important;
}}
.sb-avatar-button:hover {{ background: #D5E8DC !important; }}
.sb-menu {{
  position: absolute !important; top: 52px; right: 0; z-index: 20; width: 200px !important; min-width: 0 !important;
  padding: 8px !important; gap: 2px !important; box-shadow: 0 8px 24px rgb(28 27 24 / 0.14) !important;
}}
.sb-menu-item {{
  justify-content: flex-start !important; border: none !important; background: transparent !important; box-shadow: none !important;
  border-radius: 10px !important; padding: 10px 12px !important; font-size: 15px !important; font-weight: 500 !important;
}}
.sb-menu-item:hover {{ background: var(--sb-hover) !important; }}
.sb-back-button {{
  align-self: flex-start; width: auto !important; min-width: 0 !important; flex: none !important; padding: 8px 4px !important;
  border: none !important; background: transparent !important; box-shadow: none !important;
  color: var(--sb-accent-dark) !important; font-size: 15px !important; font-weight: 600 !important;
}}

/* Subcategories: a group per category under its marker; each row renames, moves or removes one. */
.sb-group-head {{ margin-top: 8px; }}
/* The subcategory pop-up: a dimmed backdrop over the whole page, the card centred and scrolling if tall. */
.sb-modal {{
  position: fixed !important; inset: 0; z-index: 1000; background: rgba(28, 27, 24, 0.45);
  display: flex; align-items: center; justify-content: center; padding: 16px !important; overflow-y: auto;
}}
.sb-modal-card {{ width: 100%; max-width: {PAGE_MAX_WIDTH}; max-height: 100%; overflow-y: auto; flex: none !important; }}
.sb-sub-row {{ gap: 8px !important; }}
.sb-preset-tag {{ font-size: 14px; color: var(--sb-muted); margin-top: -4px; }}
/* The plan chooser: each preset's shares, why, and the research behind it. */
.sb-preset-split {{ font-size: 14px; font-weight: 500; font-variant-numeric: tabular-nums; margin-top: 2px; }}
.sb-source {{ font-size: 14px; color: var(--sb-muted); font-style: italic; margin-top: 4px; }}
.sb-split-row {{ display: grid; gap: 8px; justify-items: start; }}
.sb-split-row > .sb-meter-row {{ width: 100%; }}
.sb-head .sb-title {{ flex-wrap: wrap; row-gap: 6px; min-width: 0; }}
.sb-title-name {{ overflow-wrap: anywhere; }}

/* Hero: the month, one big number, where the money went, and the duck. */
.sb-hero-top {{ align-items: center !important; flex-wrap: nowrap !important; }}
.sb-month-label {{ font-size: 16px; font-weight: 600; text-align: center; }}
/* Where the money went: the donut, its key beside it, stacked under it on a phone. */
.sb-donut-row {{ display: flex; align-items: center; gap: 24px; margin: 20px 0 4px; }}
.sb-donut-row .sb-key {{ flex-direction: column; }}
@media (max-width: 420px) {{
  .sb-donut-row {{ flex-direction: column; gap: 16px; }}
  .sb-donut-row .sb-key {{ flex-direction: row; justify-content: center; }}
}}
.sb-donut {{ position: relative; flex: none; }}
.sb-donut svg {{ display: block; }}
.sb-donut-middle {{
  position: absolute; inset: 0; display: flex; flex-direction: column; align-items: center; justify-content: center;
}}
.sb-donut-amount {{ font-family: var(--sb-serif); font-size: 26px; line-height: 1; font-variant-numeric: tabular-nums; }}
.sb-donut-label {{ font-size: 14px; color: var(--sb-muted); margin-top: 4px; }}
.sb-key {{ display: flex; flex-wrap: wrap; gap: 6px 16px; font-size: 14px; color: var(--sb-muted); }}
.sb-key span {{ display: inline-flex; align-items: center; gap: 6px; }}
.sb-key .sb-money {{ color: var(--sb-ink); }}
.sb-duck-says {{ display: flex; align-items: center; gap: 12px; margin-top: 20px; padding-top: 16px; border-top: 1px solid var(--sb-line); }}
.sb-bubble {{
  position: relative; background: var(--sb-paper); border-radius: 14px; padding: 10px 14px; font-size: 14px; line-height: 1.4;
}}
.sb-bubble::before {{
  content: ""; position: absolute; left: -6px; top: 50%; width: 12px; height: 12px; background: var(--sb-paper);
  transform: translateY(-50%) rotate(45deg); border-radius: 2px;
}}
.sb-duck {{ flex: none; display: block; }}
.sb-bob {{ animation: sb-bob 3.2s ease-in-out infinite; transform-origin: 50% 90%; }}
@keyframes sb-bob {{ 0%, 100% {{ transform: translateY(0) rotate(0); }} 50% {{ transform: translateY(-3px) rotate(-2deg); }} }}
@media (prefers-reduced-motion: reduce) {{ .sb-bob {{ animation: none; }} }}

/* A line in a section: name, amount, edit, delete. Dividers run the full width. */
.sb-item {{ border-top: 1px solid var(--sb-line); padding-top: 10px; }}
.sb-line {{ display: flex; justify-content: space-between; align-items: center; gap: 12px; font-size: 16px; }}
.sb-line-name {{ overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }}

/* An investment tier's heading inside the Investments card: name and subtotal. */
.sb-tier {{
  display: flex; justify-content: space-between; gap: 12px; margin-top: 6px; padding: 6px 12px; border-radius: 10px;
  background: var(--sb-paper); font-size: 14px; font-weight: 600;
}}
.sb-tier-box input {{ font-size: 16px !important; }}

/* Meters: thin, colour in the fill, words underneath. */
.sb-meter {{ height: 6px; background: var(--sb-line); border-radius: 999px; overflow: hidden; }}
.sb-meter > span {{ display: block; height: 100%; border-radius: 999px; }}
.sb-meter-row {{ display: grid; gap: 6px; }}
.sb-split {{ display: grid; gap: 16px; margin-top: 16px; }}
.sb-meter-label {{ font-size: 14px; display: flex; justify-content: space-between; gap: 8px; }}
.sb-meter-label > span:first-child {{ color: var(--sb-muted); }}

/* Buttons: pills. "+ Add" is a small green pill; edit, delete and the month arrows are quiet circles. */
.sb-add-button {{ min-width: 0 !important; padding: 8px 16px !important; font-size: 14px !important; height: 36px; flex: none !important; }}
.sb-icon {{
  border-radius: 999px !important; min-width: 40px !important; max-width: 40px; height: 40px; padding: 0 !important;
  border: none !important; background: transparent !important; color: var(--sb-muted) !important; font-size: 16px !important;
  flex: none !important;
}}
.sb-icon:hover {{ background: var(--sb-hover) !important; color: var(--sb-ink) !important; }}
.sb-nav {{ border: 1px solid var(--sb-line) !important; font-size: 20px !important; color: var(--sb-ink) !important; }}
.sb-panel {{ background: var(--sb-accent-soft); border-radius: 14px; padding: 14px !important; gap: 12px !important; }}
/* Gradio groups side-by-side boxes into one joined "form"; keep them as separate boxes on a clear background. */
.form {{ gap: 12px !important; background: transparent !important; border: none !important; box-shadow: none !important; overflow: visible !important; }}
.form > .block, .sb-panel .block {{ background: transparent !important; }}
/* Paired buttons share the row evenly; Gradio's 160px minimum would push the second one out on a phone. */
.sb-card .row > button.lg:not(.sb-icon):not(.sb-add-button) {{
  min-width: 0 !important; flex: 1 1 0 !important; padding-left: 12px !important; padding-right: 12px !important; white-space: nowrap;
}}
.sb-title + .sb-note {{ margin-top: 6px; }}

/* Tabs: a pill switcher. The tab you are on is a white pill in ink. */
.tab-wrapper {{ height: auto !important; margin-bottom: 16px; }}
.tab-container:not(.visually-hidden) {{
  height: auto !important; width: max-content; gap: 4px; padding: 4px; border-radius: 999px; background: var(--sb-paper-deep);
}}
.tab-container button {{
  height: auto !important; padding: 8px 22px !important; border: none !important; border-radius: 999px !important;
  font-size: 15px !important; font-weight: 600 !important; color: var(--sb-muted) !important; background: transparent !important;
}}
.tab-container button.selected {{ background: var(--sb-card) !important; color: var(--sb-ink) !important; box-shadow: 0 1px 3px rgb(28 27 24 / 0.12); }}
.tab-container button::after, .tab-container button.selected::after {{ display: none !important; }}

/* Filter pills: a Gradio radio styled like the tab switcher. */
/* One line always; on a narrow screen the row scrolls sideways inside itself rather than wrapping. */
.sb-pills .wrap {{
  display: flex; flex-wrap: nowrap; gap: 4px; padding: 4px; border-radius: 999px; background: var(--sb-paper-deep);
  width: max-content; max-width: 100%; overflow-x: auto; scrollbar-width: none;
}}
.sb-pills .wrap::-webkit-scrollbar {{ display: none; }}
.sb-pills label {{ flex: none; white-space: nowrap; }}
.sb-pills label {{
  margin: 0 !important; padding: 7px 16px !important; border: none !important; border-radius: 999px !important; background: transparent !important;
  font-size: 14px !important; font-weight: 600; color: var(--sb-muted) !important; box-shadow: none !important; cursor: pointer;
}}
.sb-pills label.selected {{ background: var(--sb-card) !important; color: var(--sb-ink) !important; box-shadow: 0 1px 3px rgb(28 27 24 / 0.12) !important; }}
.sb-pills input[type=radio] {{ display: none !important; }}
.sb-pills label > span {{ margin: 0 !important; }}

/* History: stat tiles, the column chart, its legend and table. */
.sb-tiles {{ display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; }}
.sb-tiles-4 {{ grid-template-columns: repeat(4, 1fr); }}
@media (max-width: 480px) {{ .sb-tiles-4 {{ grid-template-columns: repeat(2, 1fr); }} }}
.sb-tile {{ background: var(--sb-card); border: 1px solid var(--sb-line); border-radius: 16px; padding: 14px 16px; }}
.sb-tile-value {{ font-size: 24px; font-weight: 600; margin-top: 4px; letter-spacing: -0.01em; }}
.sb-tile-value.sb-over {{ color: var(--sb-over); }}
@media (max-width: 480px) {{
  .sb-tiles {{ gap: 8px; }}
  .sb-tile {{ padding: 12px; }}
  .sb-tile-value {{ font-size: 18px; }}
}}
.sb-chart {{ margin-top: 20px; }}
.sb-plot {{ position: relative; height: 220px; margin: 8px 0 0 52px; }}
.sb-grid {{ position: absolute; left: 0; right: 0; border-top: 1px solid var(--sb-line); }}
.sb-grid > span {{
  position: absolute; right: calc(100% + 8px); transform: translateY(-50%); font-size: 12px; color: var(--sb-muted);
  white-space: nowrap; font-variant-numeric: tabular-nums;
}}
.sb-cols {{ position: absolute; inset: 0; display: grid; align-items: end; }}
.sb-month-col {{ position: relative; display: flex; justify-content: center; align-items: flex-end; gap: 2px; height: 100%; min-width: 0; }}
.sb-month-col:hover, .sb-month-col:focus-visible {{ z-index: 2; }}
.sb-stack {{ position: relative; width: 24px; max-width: 42%; display: flex; flex-direction: column-reverse; gap: 2px; }}
.sb-seg {{ min-height: 2px; outline-offset: 1px; }}
.sb-seg-top {{ border-radius: 4px 4px 0 0; }}
.sb-seg:hover {{ filter: brightness(1.12); }}
.sb-zoomable {{ cursor: pointer; }}
/* One tooltip per month, beside the column so it stays inside the plot; right-half months open to the left. */
.sb-tip {{
  display: none; position: absolute; top: 0; left: calc(50% + 18px); z-index: 3; width: max-content; max-width: 220px;
  padding: 10px 12px; border-radius: 12px; background: var(--sb-card); border: 1px solid var(--sb-line);
  box-shadow: 0 4px 16px rgb(28 27 24 / 0.14); font-size: 14px; pointer-events: none; text-align: left;
}}
.sb-tip-left {{ left: auto; right: calc(50% + 18px); }}
.sb-month-col:hover .sb-tip, .sb-month-col:focus-visible .sb-tip {{ display: block; }}
/* Phones are too narrow to open beside a middle month: pin the card to the plot's far top corner instead. */
@media (max-width: 480px) {{
  .sb-month-col {{ position: static; }}
  .sb-tip {{ left: auto; right: 0; }}
  .sb-tip-left {{ left: 0; right: auto; }}
}}
.sb-tip-month {{ font-weight: 600; margin-bottom: 6px; }}
.sb-tip-row {{ display: flex; align-items: center; gap: 8px; line-height: 1.6; }}
.sb-tip-row i {{ flex: none; width: 10px; height: 10px; border-radius: 3px; }}
.sb-tip-row span {{ flex: 1; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; color: var(--sb-muted); }}
.sb-tip-row b {{ font-weight: 600; font-variant-numeric: tabular-nums; }}
.sb-tip-total {{ border-top: 1px solid var(--sb-line); margin-top: 4px; padding-top: 4px; }}
.sb-back {{ margin-top: 8px; padding: 0; border: none; background: none; font-size: 14px; font-weight: 600; color: var(--sb-accent-dark); cursor: pointer; }}
.sb-callout {{ font-size: 14px; font-weight: 600; margin-top: 8px; }}
.sb-xaxis {{ display: grid; margin: 8px 0 0 52px; font-size: 12px; color: var(--sb-muted); text-align: center; }}
.sb-xaxis span {{ white-space: nowrap; }}
.sb-hide-desktop {{ visibility: hidden; }}
@media (max-width: 480px) {{
  .sb-hide-desktop {{ visibility: visible; }}
  .sb-hide-phone {{ visibility: hidden; }}
}}
.sb-legend {{ display: flex; flex-wrap: wrap; gap: 6px 16px; margin-top: 16px; font-size: 14px; color: var(--sb-muted); }}
.sb-legend i {{ display: inline-block; width: 10px; height: 10px; border-radius: 3px; margin-right: 6px; }}
.sb-legend button {{ padding: 0; border: none; background: none; font: inherit; color: inherit; cursor: pointer; }}
.sb-legend button:hover {{ color: var(--sb-ink); }}
.sb-hidden {{ display: none !important; }}
.sb-details {{ margin-top: 16px; border-top: 1px solid var(--sb-line); padding-top: 12px; }}
.sb-details summary {{ cursor: pointer; font-size: 14px; font-weight: 600; color: var(--sb-accent-dark); }}
.sb-table-wrap {{ overflow-x: auto; }}
.sb-table, .sb-table tr {{ border: none !important; }}
.sb-table td, .sb-table th {{ white-space: nowrap; }}
.sb-table {{ width: 100%; border-collapse: collapse; margin-top: 12px; font-size: 14px; font-variant-numeric: tabular-nums; }}
.sb-table th, .sb-table td {{ text-align: right !important; padding: 8px 4px !important; border: none !important; border-bottom: 1px solid var(--sb-line) !important; background: none !important; }}
.sb-table th {{ color: var(--sb-muted); font-weight: 500; }}
.sb-table th:first-child, .sb-table td:first-child {{ text-align: left !important; }}
.sb-top-items {{ margin-top: 8px; }}
.sb-item-row {{
  display: grid; grid-template-columns: 1fr auto auto; gap: 12px; align-items: baseline; padding: 10px 0;
  border-top: 1px solid var(--sb-line); font-size: 16px;
}}

/* Sign in: the full mallard on top, everything centred. */
.sb-auth {{ max-width: 440px; margin: 24px auto 0 !important; }}
.sb-auth-top {{ display: flex; flex-direction: column; align-items: center; text-align: center; gap: 8px; }}
.sb-auth-top .sb-wordmark {{ font-size: 40px; margin-top: 4px; }}
.sb-auth-top .sb-note {{ font-size: 16px; }}
.sb-full {{ width: 100% !important; }}
.sb-danger {{ background: transparent !important; box-shadow: none !important; border-style: dashed !important; }}
"""


# ---------- The mallard ----------

# Detailed and colourful, for the sign-in card. Faces right.
DUCK_FULL = f"""<svg class="sb-duck sb-bob" viewBox="0 0 220 170" width="{{size}}" height="{{size_h}}" role="img" aria-label="Mallard duck mascot">
<defs>
  <linearGradient id="sbHead" x1="0" y1="0" x2="1" y2="1">
    <stop offset="0" stop-color="#2FA36B"/><stop offset="0.45" stop-color="{DUCK_HEAD}"/><stop offset="0.8" stop-color="#0E3B2A"/><stop offset="1" stop-color="#3A2F6B"/>
  </linearGradient>
  <linearGradient id="sbBody" x1="0" y1="0" x2="0" y2="1">
    <stop offset="0" stop-color="#D4CFC5"/><stop offset="1" stop-color="#A39D91"/>
  </linearGradient>
  <linearGradient id="sbBreast" x1="0" y1="0" x2="1" y2="1">
    <stop offset="0" stop-color="#9A5430"/><stop offset="1" stop-color="{DUCK_BREAST}"/>
  </linearGradient>
  <linearGradient id="sbWing" x1="0" y1="0" x2="1" y2="0">
    <stop offset="0" stop-color="#7F57C9"/><stop offset="1" stop-color="#2F4FC4"/>
  </linearGradient>
</defs>
<ellipse cx="110" cy="152" rx="92" ry="10" fill="#D9ECF2"/>
<path d="M44 150 q12 -6 24 0 q12 6 24 0 M128 152 q12 -6 24 0 q12 6 24 0" fill="none" stroke="#B9DCE7" stroke-width="2.5" stroke-linecap="round"/>
<path d="M92 144 l-7 10 l13 -2 z M118 145 l-5 10 l12 -3 z" fill="#E88B2E"/>
<path d="M20 93 L38 85 L36 100 Z" fill="#F4F2ED" stroke="#D8D3C9" stroke-width="1"/>
<path d="M28 90 C46 84 70 80 104 80 C132 80 150 84 160 92 C176 104 178 126 160 138 C146 146 110 147 80 145 C58 143 42 132 34 116 C30 108 28 98 28 90 Z" fill="url(#sbBody)"/>
<path d="M28 90 C38 86 48 84 58 85 C55 94 47 101 36 105 C31 100 28 95 28 90 Z" fill="#26241F"/>
<path d="M47 86 C40 77 44 69 51 70 C56 71 56 78 50 79" fill="none" stroke="#26241F" stroke-width="3.5" stroke-linecap="round"/>
<path d="M70 128 C92 134 124 134 150 128 M62 135 C88 141 126 141 148 135" fill="none" stroke="#9D978B" stroke-width="1.1" stroke-linecap="round" opacity="0.6"/>
<path d="M134 86 C158 84 178 100 178 118 C178 130 168 138 156 138 C160 124 154 102 134 86 Z" fill="url(#sbBreast)"/>
<path d="M130 96 C112 87 72 86 46 99 C60 113 94 121 122 116 C134 112 138 102 130 96 Z" fill="#8F887C"/>
<path d="M60 106 C72 107 86 109 98 108 C90 115 72 116 57 111 Z" fill="url(#sbWing)"/>
<path d="M60 106 C72 107 86 109 98 108 M57 111 C72 116 90 115 98 108" fill="none" stroke="#FFFFFF" stroke-width="1.6" stroke-linecap="round"/>
<path d="M80 96 C96 92 112 93 124 99 M72 101 C92 96 112 98 126 105 M104 112 C112 112 120 110 126 106" fill="none" stroke="#7A7468" stroke-width="1.3" stroke-linecap="round"/>
<path d="M135 95 C133 79 139 65 151 58 L179 62 C179 78 173 91 163 99 Z" fill="url(#sbHead)"/>
<ellipse cx="163" cy="50" rx="27" ry="23" fill="url(#sbHead)"/>
<ellipse cx="155" cy="40" rx="10" ry="4.5" fill="#FFFFFF" opacity="0.22" transform="rotate(-20 155 40)"/>
<path d="M136 93 C146 100 158 100 167 93" fill="none" stroke="#FFFFFF" stroke-width="4.5" stroke-linecap="round"/>
<path d="M183 47 C196 44 208 49 210 56 C205 62 193 62 182 59 Z" fill="{DUCK_BILL}"/>
<path d="M184 54 C194 55 202 55 208 55" fill="none" stroke="#C9971F" stroke-width="1.4" stroke-linecap="round"/>
<circle cx="208.5" cy="55.5" r="1.8" fill="#3A2A10"/>
<circle cx="173" cy="44" r="3.6" fill="#121110"/>
<circle cx="174.3" cy="42.7" r="1.2" fill="#FFFFFF"/>
</svg>"""

# Simplified for the logo and the duck's speech bubble: flat colours, readable at 24px. Faces right.
DUCK_MARK = f"""<svg class="sb-duck" viewBox="0 0 64 64" width="{{size}}" height="{{size}}" role="img" aria-label="Sliced Bread duck">
<path d="M5 33 C14 29 22 28 32 28 C44 28 53 33 55 41 C57 50 49 56 38 56 L20 56 C11 56 6 47 5 33 Z" fill="{DUCK_BODY}"/>
<path d="M42 31 C52 33 58 41 56 48 C54 53 48 55 44 53 C47 46 47 37 42 31 Z" fill="{DUCK_BREAST}"/>
<path d="M13 31 C9 28 10 23 14 23" fill="none" stroke="#26241F" stroke-width="2.6" stroke-linecap="round"/>
<path d="M40 31 C40 24 43 20 47 19 L53 22 C53 27 51 31 47 34 Z" fill="{DUCK_HEAD}"/>
<circle cx="47" cy="17" r="11" fill="{DUCK_HEAD}"/>
<path d="M40 31 C43.5 33.5 47.5 33.5 51 30.5" fill="none" stroke="#FFFFFF" stroke-width="2.4" stroke-linecap="round"/>
<path d="M56 14.5 C60.5 13.5 63.5 15.5 63.5 18 C61.5 20 58.5 20 56 19 Z" fill="{DUCK_BILL}"/>
<circle cx="50.5" cy="14" r="1.8" fill="#121110"/>
</svg>"""


def duck_full(size=160):
    return DUCK_FULL.replace("{size}", str(size)).replace("{size_h}", str(round(size * 170 / 220)))


def duck_mark(size=32):
    return DUCK_MARK.replace("{size}", str(size))


# ---------- HTML pieces (escape everything the user typed) ----------

def shades(color, count, darkest=DARKEST_MIX, lightest=LIGHTEST_MIX):
    """count shades of one hex colour, darkest first: the History zoom's biggest line to its smallest."""
    if count == 1:
        return [color]
    red, green, blue = (int(color[i:i + 2], 16) for i in (1, 3, 5))
    result = []
    for index in range(count):
        mix = -darkest + (darkest + lightest) * index / (count - 1)  # below 0 toward black, above toward white
        target = 255 if mix > 0 else 0
        channels = (round(value + (target - value) * abs(mix)) for value in (red, green, blue))
        result.append("#" + "".join(f"{channel:02X}" for channel in channels))
    return result


def subcategory_colors(user_budget):
    """Each subcategory's colour: a shade of its category's, in page order."""
    colors = {}
    for category, color in CATEGORY_COLORS.items():
        keys = [sub["key"] for sub in budget.subcategories_in(user_budget, category)]
        colors.update(zip(keys, shades(color, len(keys), SUBCATEGORY_DARKEST_MIX, SUBCATEGORY_LIGHTEST_MIX)))
    return colors


def donut_svg(amounts, income):
    """The hero ring: one segment per category with money in it, income in the middle. A grey ring when empty."""
    radius = (DONUT_SIZE - DONUT_STROKE) / 2
    circumference = 2 * math.pi * radius
    centre = DONUT_SIZE / 2
    whole = sum(amounts.values())
    parts = [(category, cents) for category, cents in amounts.items() if cents]
    ring = (
        f'<circle cx="{centre}" cy="{centre}" r="{radius}" fill="none" stroke="{LINE}" stroke-width="{DONUT_STROKE}"></circle>'
    )
    start = 0.0
    segments = ""
    for category, cents in parts:
        length = cents / whole * circumference
        gap = DONUT_GAP if len(parts) > 1 else 0
        title = f"{budget.CATEGORY_NAMES[category]} {budget.format_money(cents)} ({round(cents * 100 / whole)}%)"
        segments += (
            f'<circle cx="{centre}" cy="{centre}" r="{radius}" fill="none" stroke="{CATEGORY_COLORS[category]}" '
            f'stroke-width="{DONUT_STROKE}" stroke-dasharray="{max(length - gap, 0):.2f} {circumference:.2f}" '
            f'stroke-dashoffset="{-start:.2f}" transform="rotate(-90 {centre} {centre})"><title>{title}</title></circle>'
        )
        start += length
    return (
        f'<div class="sb-donut"><svg width="{DONUT_SIZE}" height="{DONUT_SIZE}" viewBox="0 0 {DONUT_SIZE} {DONUT_SIZE}" '
        f'role="img" aria-label="Where the money went">{ring if not parts else ""}{segments}</svg>'
        f'<div class="sb-donut-middle"><span class="sb-donut-amount">{budget.format_money(income)}</span>'
        '<span class="sb-donut-label">income</span></div></div>'
    )


def dot(color):
    return f'<span class="sb-dot" style="background:{color}"></span>'


def marker(category):
    """Which of the four categories something counts toward: a small pill with the category's dot and name."""
    return f'<span class="sb-marker">{dot(CATEGORY_COLORS[category])}{budget.CATEGORY_NAMES[category]}</span>'


def money(cents):
    return f'<span class="sb-money">{budget.format_money(cents)}</span>'


def limit_color(fraction):
    """A spending limit's fill: green when empty, shifting through amber to red at the limit."""
    fraction = min(max(fraction, 0.0), 1.0)
    for (start, low), (end, high) in zip(LIMIT_COLOR_STOPS, LIMIT_COLOR_STOPS[1:]):
        if fraction <= end:
            share = (fraction - start) / (end - start)
            channels = (
                round(int(low[i:i + 2], 16) + (int(high[i:i + 2], 16) - int(low[i:i + 2], 16)) * share) for i in (1, 3, 5)
            )
            return "#" + "".join(f"{channel:02X}" for channel in channels)
    return OVER


def meter_html(result, label_left="", blend=False):
    """A meter with its words; blend for spending limits, which redden as they fill. Empty when no limit is set."""
    if result["state"] == "none":
        return ""
    width = max(result["fraction"] * 100, 2)
    color = limit_color(result["fraction"]) if blend else METER_COLORS[result["state"]]
    return (
        f'<div class="sb-meter-row"><div class="sb-meter"><span style="width:{width:.1f}%;'
        f'background:{color}"></span></div>'
        f'<div class="sb-meter-label"><span>{html.escape(label_left)}</span><span>{html.escape(result["label"])}</span></div></div>'
    )


def progress_html(saved, goal):
    """The savings goal bar: always green, it only fills."""
    width = max(min(saved / goal, 1) * 100, 2)
    return (
        f'<div class="sb-meter-row"><div class="sb-meter"><span style="width:{width:.1f}%;background:{ACCENT}"></span></div>'
        f'<div class="sb-meter-label"><span>Savings goal</span><span class="sb-money">'
        f'{budget.format_money(saved)} of {budget.format_money(goal)}</span></div></div>'
    )


def line_html(name, cents):
    return f'<div class="sb-line"><span class="sb-line-name">{html.escape(name)}</span>{money(cents)}</div>'


## `charts.py`

The History tab's chart, tiles and table: categories, then a subcategory, then its items. Plain HTML.

In [ ]:
%%writefile charts.py
"""History charts for Sliced Bread: earned vs money out by month, zooming into a category, then a subcategory.
HTML only, no Gradio.

Built from plain HTML and CSS columns (not SVG) so labels stay readable at phone width. Follows the dataviz
rules: columns at most 24px wide with 4px rounded tops, 2px gaps between stacked pieces, hairline gridlines,
a legend, one callout above the plot, one hover/keyboard card per month, and a table with every number.
"""

from html import escape

import budget
import design

NICE_STEPS = (1, 2, 2.5, 5)  # tick steps, times a power of ten
TARGET_TICKS = 4  # gridlines above zero
HEADROOM = 1.12  # space above the tallest column for its label
COMPACT_FROM_CENTS = 10_000 * budget.CENTS_PER_DOLLAR  # axis switches to "$10K" style from here
# Month labels: show every Nth, counted back from the latest month, so they never run together.
LABEL_EVERY = ((12, 1), (24, 2))  # desktop: up to 12 months label all, up to 24 every other
PHONE_LABEL_EVERY = ((6, 1), (12, 2), (24, 4))  # phone widths fit about six labels
MAX_SHADED_ITEMS = 5  # lines named in a zoomed chart; smaller ones share one Other piece
OTHER_LABEL = "Other"
ALL_CHOICE = budget.ALL_SPENDING  # the zoom radio's label for the unzoomed chart
LEFT_OVER_LABEL = "Left over"  # what the Savings zoom adds on top of its subcategories
TOP_ITEMS = 5  # biggest lines listed under a zoomed chart


def nice_ceiling(cents):
    """The top of the y-axis and its step: a clean number just above the tallest column."""
    target = max(cents * HEADROOM, budget.CENTS_PER_DOLLAR) / TARGET_TICKS
    magnitude = 10 ** (len(str(int(target))) - 1)
    step = int(next(m * magnitude for m in NICE_STEPS + (10,) if m * magnitude >= target))
    return step * TARGET_TICKS, step


def tick_label(cents):
    dollars = cents / budget.CENTS_PER_DOLLAR
    if cents >= COMPACT_FROM_CENTS:
        return f"${dollars / 1000:g}K"
    return f"${dollars:,.0f}"


def full_month(key):
    return budget.month_label(key)


# ---------- What a zoom level shows ----------

def zoom_choices(user_budget):
    """The hidden zoom radio's (label, value) choices. Clicking a piece picks the choice whose label it names."""
    return (
        [(ALL_CHOICE, "all")]
        + [(budget.CATEGORY_NAMES[category], f"category:{category}") for category in budget.CATEGORIES]
        + [(sub["name"], f"sub:{sub['key']}") for sub in budget.subcategories(user_budget)]
    )


def zoom_series(lines, color):
    """The zoomed subcategory's lines ranked by their range total, biggest first, as [(label, color)]; the tail is Other."""
    totals = {}
    for month_lines in lines.values():
        for label, cents in month_lines.items():
            totals[label] = totals.get(label, 0) + cents
    ranked = sorted(totals, key=lambda label: totals[label], reverse=True)
    names = ranked[:MAX_SHADED_ITEMS] + ([OTHER_LABEL] if len(ranked) > MAX_SHADED_ITEMS else [])
    return list(zip(names, design.shades(color, len(names), design.DARKEST_MIX, design.ITEM_LIGHTEST_MIX))) if names else []


def month_amounts(month_lines, series):
    """One month's cents per series entry; lines outside the named ones add up into Other."""
    named = {label for label, _ in series if label != OTHER_LABEL}
    other = sum(cents for label, cents in month_lines.items() if label not in named)
    return [other if label == OTHER_LABEL else month_lines.get(label, 0) for label, _ in series]


def chart_view(user_budget, rows, pick):
    """What the chart shows for a zoom pick ("all", "category:needs", "sub:travel").

    series is [(name, color, zoom_to, [cents per row])], stacked bottom first. zoom_to names the choice a click opens.
    """
    kind, _, key = pick.partition(":")
    if kind == "category":
        colors = design.subcategory_colors(user_budget)
        subs = budget.subcategories_in(user_budget, key)
        series = [(sub["name"], colors[sub["key"]], sub["name"], [row["subcategories"][sub["key"]] for row in rows]) for sub in subs]
        if key == "savings":
            lightest = design.shades(design.CATEGORY_COLORS[key], 2, 0, design.LIGHTEST_MIX)[-1]
            series.append((LEFT_OVER_LABEL, lightest, None, [max(row["left"], 0) for row in rows]))
        return {"level": "category", "category": key, "name": budget.CATEGORY_NAMES[key], "series": series,
                "back": ALL_CHOICE, "sections": [sub["key"] for sub in subs]}
    if kind == "sub":
        sub = next(sub for sub in budget.subcategories(user_budget) if sub["key"] == key)
        lines = budget.section_lines(user_budget, [row["key"] for row in rows], key)
        shaded = zoom_series(lines, design.subcategory_colors(user_budget)[key])
        per_row = [month_amounts(lines[row["key"]], shaded) for row in rows]
        series = [(label, color, None, [amounts[index] for amounts in per_row]) for index, (label, color) in enumerate(shaded)]
        return {"level": "sub", "category": sub["category"], "name": sub["name"], "series": series,
                "back": budget.CATEGORY_NAMES[sub["category"]], "sections": [key]}
    series = [
        (budget.CATEGORY_NAMES[category], design.CATEGORY_COLORS[category], budget.CATEGORY_NAMES[category],
         [row["categories"][category] for row in rows])
        for category in budget.CATEGORIES
    ]
    return {"level": "all", "category": None, "name": ALL_CHOICE, "series": series, "back": None, "sections": []}


def view_totals(view, rows):
    """Each month's stacked total for a view."""
    return [sum(values[index] for _, _, _, values in view["series"]) for index in range(len(rows))]


# ---------- The chart ----------

def piece(value, color, zoom_to, is_top):
    """One stacked piece. The month's tooltip carries the numbers; a piece can zoom the chart to what it shows."""
    classes = "sb-seg" + (" sb-seg-top" if is_top else "") + (" sb-zoomable" if zoom_to else "")
    zoom = f' data-sb-zoom="{escape(zoom_to)}"' if zoom_to else ""
    return f'<div class="{classes}" style="flex:{value};background:{color}"{zoom}></div>'


def column(pieces, height_percent):
    """A column of stacked (value, color, zoom_to) pieces, bottom piece first, as tall as height_percent of the plot."""
    shown = [p for p in pieces if p[0] > 0]
    if not shown:
        return ""
    inner = "".join(piece(value, color, zoom_to, index == len(shown) - 1) for index, (value, color, zoom_to) in enumerate(shown))
    return f'<div class="sb-stack" style="height:{height_percent:.2f}%">{inner}</div>'


def tooltip(month, entries, total, opens_left):
    """The month's hover card: a swatch, name and amount per (name, color, cents), then the (name, cents) total."""
    rows = "".join(
        f'<div class="sb-tip-row"><i style="background:{color}"></i><span>{escape(name)}</span><b>{budget.format_money(cents)}</b></div>'
        for name, color, cents in entries
    )
    total_row = f'<div class="sb-tip-row sb-tip-total"><i></i><span>{total[0]}</span><b>{budget.format_money(total[1])}</b></div>'
    side = " sb-tip-left" if opens_left else ""
    return f'<div class="sb-tip{side}" aria-hidden="true"><div class="sb-tip-month">{month}</div>{rows}{total_row}</div>'


def month_column(month, stacks, entries, total, opens_left):
    """One month: its stacked columns plus the tooltip, focusable so the keyboard gets the same card."""
    summary = ", ".join(f"{name} {budget.format_money(cents)}" for name, _, cents in entries)
    label = escape(f"{month}: {summary}, {total[0]} {budget.format_money(total[1])}")
    return f'<div class="sb-month-col" tabindex="0" aria-label="{label}">{"".join(stacks)}{tooltip(month, entries, total, opens_left)}</div>'


def history_chart(rows, view):
    """Earned vs money out per month, or one category or subcategory split into its parts, bottom piece first."""
    zoomed = view["level"] != "all"
    totals = view_totals(view, rows)
    tallest = max(totals) if zoomed else max(max(row["income"], total) for row, total in zip(rows, totals))
    top, step = nice_ceiling(tallest)

    count = len(rows)
    columns = []
    for index, (row, total) in enumerate(zip(rows, totals)):
        pieces = [(values[index], color, zoom_to) for _, color, zoom_to, values in view["series"]]
        entries = [(name, color, values[index]) for name, color, _, values in view["series"] if values[index]]
        stacks = [column(pieces, total / top * 100)]
        if not zoomed:
            stacks.insert(0, column([(row["income"], design.INCOME_COLOR, None)], row["income"] / top * 100))
            entries.insert(0, ("Earned", design.INCOME_COLOR, row["income"]))
        total_label = "Total" if zoomed else "Money out"
        columns.append(month_column(full_month(row["key"]), stacks, entries, (total_label, total), index >= count / 2))

    grid = "".join(
        f'<div class="sb-grid" style="bottom:{value / top * 100:.2f}%"><span>{tick_label(value)}</span></div>'
        for value in range(0, top + 1, step)
    )
    every = next(step for limit, step in LABEL_EVERY if count <= limit)
    phone_every = next(step for limit, step in PHONE_LABEL_EVERY if count <= limit)
    months = "".join(
        f'<span class="{label_classes(count - 1 - index, every, phone_every)}">{row["label"]}</span>'
        for index, row in enumerate(rows)
    )
    title = f"{view['name']} by month" if zoomed else "Earned vs money out"
    # A zoomed subcategory carries its category's marker; a category is its own marker.
    marker = design.marker(view["category"]) if view["level"] == "sub" else ""
    span = full_month(rows[0]["key"]) if count == 1 else f"{full_month(rows[0]['key'])} to {full_month(rows[-1]['key'])}"
    back = f'<button class="sb-back" data-sb-zoom="{escape(view["back"])}">&larr; {escape(view["back"])}</button>' if zoomed else ""
    can_zoom = any(zoom_to for _, _, zoom_to, _ in view["series"])
    hint = '<div class="sb-note">Click a colour to see what it is made of.</div>' if can_zoom else ""
    return (
        f'<div class="sb-title">{escape(title)}{marker}</div><div class="sb-note">{span}</div>{back}{best_month_html(rows, view)}'
        f'<div class="sb-chart" role="group" aria-label="{escape(title)}, {escape(span)}">'
        f'<div class="sb-plot">{grid}<div class="sb-cols" style="grid-template-columns:repeat({count},minmax(0,1fr))">{"".join(columns)}</div></div>'
        f'<div class="sb-xaxis" style="grid-template-columns:repeat({count},minmax(0,1fr))">{months}</div></div>'
        f"{legend_html(view)}{hint}{numbers_table_html(rows, view)}"
    )


def best_month_html(rows, view):
    """The one callout: the month that stands out, above the chart so it never sits on a column."""
    if view["level"] != "all":
        amount, best = max(zip(view_totals(view, rows), rows), key=lambda pair: pair[0])
        text = f"Biggest month: {full_month(best['key'])}, {budget.format_money(amount)}"
    else:
        best = max(rows, key=lambda row: row["saved"])
        amount = best["saved"] if best["income"] else 0
        text = f"Best month: {full_month(best['key'])}, {budget.format_money(amount)} saved"
    return f'<div class="sb-callout">{text}</div>' if amount > 0 else ""


def label_classes(months_back, every, phone_every):
    """Which screens hide this month's label: it shows only on every Nth month back from the latest."""
    hidden = []
    if months_back % every:
        hidden.append("sb-hide-desktop")
    if months_back % phone_every:
        hidden.append("sb-hide-phone")
    return " ".join(hidden)


def legend_html(view):
    """Every piece's colour and name; the ones that zoom in are buttons."""
    items = "" if view["level"] != "all" else f'<span><i style="background:{design.INCOME_COLOR}"></i>Earned</span>'
    for name, color, zoom_to, _ in view["series"]:
        swatch = f'<i style="background:{color}"></i>{escape(name)}'
        items += f'<button data-sb-zoom="{escape(zoom_to)}">{swatch}</button>' if zoom_to else f"<span>{swatch}</span>"
    return f'<div class="sb-legend">{items}</div>'


def stat_tiles_html(rows, view):
    """The range in numbers. Plain sans, proportional digits: these are standalone figures."""
    earned = sum(row["income"] for row in rows)
    if view["level"] != "all":
        total = sum(view_totals(view, rows))
        active = [row for row in rows if row["income"] or row["spent"] or row["invested"]] or rows
        share = f"{round(total * 100 / earned)}%" if earned else "0%"
        first = {"investing": "Invested", "savings": "Saved"}.get(view["category"], f"Spent on {view['name'].lower()}")
        tiles = [(first, budget.format_money(total), ""),
                 ("Monthly average", budget.format_money(total // len(active)), ""),
                 ("Share of income", share, "")]
    else:
        saved = sum(row["saved"] for row in rows)
        tiles = [("Earned", budget.format_money(earned), ""),
                 ("Spent", budget.format_money(sum(row["spent"] for row in rows)), ""),
                 ("Invested", budget.format_money(sum(row["invested"] for row in rows)), ""),
                 ("Saved", budget.format_money(saved), " sb-over" if saved < 0 else "")]
    cells = "".join(
        f'<div class="sb-tile"><div class="sb-eyebrow">{escape(label)}</div><div class="sb-tile-value{extra}">{value}</div></div>'
        for label, value, extra in tiles
    )
    return f'<div class="sb-tiles sb-tiles-{len(tiles)}">{cells}</div>'


def numbers_table_html(rows, view):
    """Every number in the chart, as a table: the version that needs no hovering."""
    if view["level"] == "all":
        columns = [("Earned", [row["income"] for row in rows]), ("Spent", [row["spent"] for row in rows]),
                   ("Invested", [row["invested"] for row in rows]), ("Saved", [row["saved"] for row in rows])]
    else:
        columns = [(name, values) for name, _, _, values in view["series"]] + [("Total", view_totals(view, rows))]
    head = "<th>Month</th>" + "".join(f"<th>{escape(name)}</th>" for name, _ in columns)
    body = "".join(
        f"<tr><td>{rows[index]['label']} {rows[index]['key'][:4]}</td>"
        + "".join(f"<td>{budget.format_money(values[index])}</td>" for _, values in columns)
        + "</tr>"
        for index in reversed(range(len(rows)))
    )
    return (
        '<details class="sb-details"><summary>See the numbers</summary>'
        f'<div class="sb-table-wrap"><table class="sb-table"><tr>{head}</tr>{body}</table></div></details>'
    )


def top_items(user_budget, rows, view):
    """The biggest lines across a zoomed view's subcategories: (name, month_key, cents), largest first."""
    keys = [row["key"] for row in rows]
    items = [item for section in view["sections"] for item in budget.top_items(user_budget, keys, section, TOP_ITEMS)]
    return sorted(items, key=lambda item: item[2], reverse=True)[:TOP_ITEMS]


def top_items_html(items, view):
    """The biggest lines in the zoomed view: the detail under the chart."""
    title = f'<div class="sb-title">Biggest in {escape(view["name"].lower())}</div>'
    if not items:
        return title + '<div class="sb-note" style="margin-top:6px">Nothing in this range yet.</div>'
    rows = "".join(
        f'<div class="sb-item-row"><span class="sb-line-name">{escape(name)}</span>'
        f'<span class="sb-note">{budget.month_name(key)[:3]} {key[:4]}</span>{design.money(cents)}</div>'
        for name, key, cents in items
    )
    return f'{title}<div class="sb-top-items">{rows}</div>'


## `app.py`

The Gradio screens: sign in, the Budget, History and Goals tabs, and the profile page. The only file that uses Gradio.

In [ ]:
%%writefile app.py
"""Sliced Bread: a personal budget tracker. Run with: python app.py"""

import copy
import sys
from html import escape

import gradio as gr

import budget
import charts
import design
import storage

INCOME_NAME = "Income"
EMPTY_STATES = {
    "income": "No income yet. Add your salary, freelance or other pay.",
    "necessities": "Nothing yet. Add rent, groceries, gas.",
    "discretionary": "Nothing yet. Add movies, events, games, subscriptions.",
    "travel": "No trips yet. Add flights, hotels, food.",
    "investments": "Nothing yet. Add a tier like Retirement, then what you put in.",
    "emergency_fund": "Nothing yet. Add what you move into savings.",
}
CUSTOM_EMPTY_STATE = "Nothing yet."
NAME_PLACEHOLDERS = {
    "income": "Salary", "necessities": "Rent", "discretionary": "Concert tickets", "travel": "Flights", "investments": "Roth IRA",
    "emergency_fund": "Transfer",
}
MONTHLY_LIMIT = "Monthly limit"
CATEGORY_PICKER_WIDTH = 124  # px: fits "Investing" and the dropdown arrow
CATEGORY_OPTIONS = [(budget.CATEGORY_NAMES[category], category) for category in budget.CATEGORIES]
RANGE_CHOICES = ["6 months", "12 months", "All"]
SAMPLE_OFFER_BELOW = 3  # months with data; under this, History offers sample months


def html(value, **options):
    return gr.HTML(value, padding=False, **options)


def fail(message):
    """Show a validation problem to the user without a traceback in the console."""
    return gr.Error(message, print_exception=False)


def changed(user, user_budget, change, *args):
    """Apply one budget change to a copy, save it and return it; a ValueError becomes a message to the user."""
    updated = copy.deepcopy(user_budget)
    try:
        change(updated, *args)
    except ValueError as error:
        raise fail(str(error))
    storage.save_budget(user, updated)
    return updated


def sections(user_budget):
    """Every section on the Budget page as (key, name, category): income, then subcategories by category."""
    return [(budget.INCOME, INCOME_NAME, None)] + [
        (sub["key"], sub["name"], category)
        for category in budget.CATEGORIES
        for sub in budget.subcategories_in(user_budget, category)
    ]


def limit_meters(user_budget, month):
    """Each limit's (meter, words) by the name alerts give it: Needs and Wants as their share of the monthly limit,
    or the monthly limit on all spending while targets are off."""
    settings = user_budget["settings"]
    shares = budget.category_limits(settings)
    if shares:
        meters = {}
        for category in budget.SPENDING_CATEGORIES:
            name, spent = budget.CATEGORY_NAMES[category], budget.category_total(user_budget, month, category)
            words = f"{name} {budget.format_money(spent)} of {budget.format_money(shares[category])}"
            meters[f"{name} limit"] = (budget.meter(spent, shares[category]), words)
        return meters
    overall = settings["limits"].get("overall", 0)
    spent = budget.total_expenses(user_budget, month)
    return {MONTHLY_LIMIT: (budget.meter(spent, overall), f"Spent {budget.format_money(spent)} of {budget.format_money(overall)}")}


# ---------- Page pieces ----------

def month_title_html(key):
    return f'<div class="sb-month-label">{budget.month_label(key)}</div>'


def person_html(user_budget):
    return f'<span class="sb-person-name">{escape(user_budget["account"]["name"])}</span>' if user_budget else ""


def initial(user_budget):
    return user_budget["account"]["name"][:1].upper() if user_budget else ""


def duck_line(user_budget, key):
    """What the duck says under the hero: one line picked from the month's state. The mascot's only job for now."""
    month = budget.get_month(user_budget, key)
    if budget.section_total(month, budget.INCOME) == 0:
        return "Add your income and I'll do the math."
    left = budget.leftover(user_budget, month)
    if left < 0:
        return "Quack. More went out than came in this month."
    meters = watched_meters(user_budget, key)
    for label, result in meters.items():
        if result["state"] == "over":
            return f"Heads up: {label.lower()} is {result['label'].lower()}."
    for label, result in meters.items():
        if result["state"] == "warn":
            return f"Careful, {label.lower()} is near its limit."
    return f"Nice. {budget.format_money(budget.saved(user_budget, month))} saved this month."


def hero_html(user_budget, key):
    """The big number, where the money went by category, and the duck."""
    month = budget.get_month(user_budget, key)
    income = budget.section_total(month, budget.INCOME)
    left = budget.leftover(user_budget, month)
    name = budget.month_name(key)

    if left >= 0:
        top = f'<div class="sb-eyebrow">Left to save in {name}</div><div class="sb-hero-amount">{budget.format_money(left)}</div>'
    else:
        top = f'<div class="sb-eyebrow">Over budget in {name}</div><div class="sb-hero-amount sb-over">{budget.format_money(-left)}</div>'

    amounts = {category: budget.category_total(user_budget, month, category) for category in budget.CATEGORIES}
    amounts["savings"] = budget.saved(user_budget, month)
    key_items = "".join(
        f"<span>{design.dot(design.CATEGORY_COLORS[category])}{budget.CATEGORY_NAMES[category]} {design.money(cents)}</span>"
        for category, cents in amounts.items() if cents
    )
    meters = "".join(design.meter_html(result, words, blend=True) for result, words in limit_meters(user_budget, month).values())
    duck = (
        f'<div class="sb-duck-says"><span class="sb-bob">{design.duck_mark(44)}</span>'
        f'<div class="sb-bubble">{escape(duck_line(user_budget, key))}</div></div>'
    )
    return (
        f'{top}<div class="sb-donut-row">{design.donut_svg(amounts, income)}<div class="sb-key">{key_items}</div></div>'
        + (f'<div class="sb-split">{meters}</div>' if meters else "")
        + duck
    )


def section_head_html(user_budget, month, section, name, category):
    """A section's title, the category it counts toward, and its total."""
    total = budget.section_total(month, section)
    color = design.INCOME_COLOR if category is None else design.subcategory_colors(user_budget)[section]
    marker = design.marker(category) if category else ""
    return (
        f'<div class="sb-head"><div class="sb-title">{design.dot(color)}<span class="sb-title-name">{escape(name)}</span>{marker}</div>'
        f'<div class="sb-head-total">{design.money(total)}</div></div>'
    )


def tier_head_html(tier, cents):
    return f'<div class="sb-tier"><span>{escape(tier)}</span>{design.money(cents)}</div>'


def tier_box(user_budget, value=""):
    """Type a tier: an existing one (the placeholder lists them) or a new name, which creates it."""
    used = budget.tiers(user_budget)
    hint = ", ".join(used) if used else "Retirement"
    # interactive=True: Gradio only enables a box by itself when an event writes to it, and the edit form's never do.
    return gr.Textbox(label="Tier", value=value, placeholder=hint, autofocus=not value, interactive=True, min_width=0, elem_classes="sb-tier-box")


def savings_html(user_budget, key):
    """The Savings category: what was set aside on purpose plus what is left over, and the savings goal."""
    month = budget.get_month(user_budget, key)
    left = budget.leftover(user_budget, month)
    set_aside = budget.total_set_aside(user_budget, month)
    head = f'<div class="sb-title">{design.dot(design.CATEGORY_COLORS["savings"])}Savings</div>'
    amount = f'<div class="sb-big" style="margin-top:12px">{budget.format_money(budget.saved(user_budget, month))}</div>'
    if left < 0:
        note = f'<div class="sb-note" style="color:var(--sb-over)">Over budget by {budget.format_money(-left)}. Nothing left over.</div>'
    elif set_aside:
        note = f'<div class="sb-note">{budget.format_money(set_aside)} set aside, {budget.format_money(left)} left over.</div>'
    else:
        note = '<div class="sb-note">Whatever is left after spending and investing lands here.</div>'
    goal = user_budget["settings"]["savings_goal"]
    if goal:
        progress = design.progress_html(budget.savings_progress(user_budget), goal)
    else:
        progress = '<div class="sb-note">Set a savings goal on the Goals tab.</div>'
    return f'{head}{amount}<div style="margin-top:6px">{note}</div><div style="margin-top:16px">{progress}</div>'


def split_html(user_budget, month):
    """Each category against the user's percentage targets, under the same marker as its sections. Empty when off."""
    targets = user_budget["settings"].get("targets", {})
    meters = budget.target_meters(user_budget, targets, month)
    if not meters:
        return ""
    plan = "/".join(str(targets[category]) for category in budget.CATEGORIES)
    rows = "".join(
        f'<div class="sb-split-row">{design.marker(category)}'
        f'{design.meter_html(result, f"{result["actual_percent"]}% of {result["target_percent"]}%", blend=category in budget.SPENDING_CATEGORIES)}</div>'
        for category, result in meters.items()
    )
    return f'<div class="sb-title">Your {plan} split</div><div class="sb-split">{rows}</div>'


def watched_meters(user_budget, key):
    """Every limit and target an alert watches, by the name the alert shows."""
    month = budget.get_month(user_budget, key)
    meters = {label: result for label, (result, _) in limit_meters(user_budget, month).items()}
    targets = budget.target_meters(user_budget, user_budget["settings"].get("targets", {}), month)
    # Only needs and wants are limits; investing and savings are minimums, so they never alert.
    meters.update({
        f"{budget.CATEGORY_NAMES[category]} target": result
        for category, result in targets.items() if category in budget.SPENDING_CATEGORIES
    })
    return meters


def budget_alerts(before, after, key):
    """Warn when a change pushes a limit or target into "near" or "over"."""
    old = watched_meters(before, key)
    for label, new in watched_meters(after, key).items():
        if new["state"] in ("warn", "over") and new["state"] != old.get(label, {"state": "none"})["state"]:
            gr.Warning(f"{label}: {new['label']}", title="Budget alert")


# ---------- Event handlers: lines ----------

def make_open(value):
    return lambda: value


def make_add(section):
    def add(user, user_budget, key, name, amount, tier=None):
        updated = changed(user, user_budget, budget.add_item, key, section, name, amount, tier)
        budget_alerts(user_budget, updated, key)
        return updated, "", ""
    return add


def make_update(section, index):
    def update(user, user_budget, key, name, amount, tier=None):
        updated = changed(user, user_budget, budget.update_item, key, section, index, name, amount, tier)
        budget_alerts(user_budget, updated, key)
        return updated, None
    return update


def make_remove(section, index):
    def remove(user, user_budget, key, open_section):
        updated = changed(user, user_budget, budget.remove_item, key, section, index)
        # An open edit row points at a line position that just moved, so close it; an add row can stay.
        return updated, open_section if open_section and open_section[0] == "add" else None
    return remove


# ---------- Event handlers: subcategories ----------

def make_confirm(keys):
    """Save every staged rename and move at once and close the pop-up. One bad name saves nothing."""
    def confirm(user, user_budget, *values):
        names, categories = values[:len(keys)], values[len(keys):]

        def apply(updated):
            for key, name, category in zip(keys, names, categories):
                budget.rename_subcategory(updated, key, name)
                budget.move_subcategory(updated, key, category)
        return changed(user, user_budget, apply), False
    return confirm


def make_remove_subcategory(key):
    def remove(user, user_budget):
        return changed(user, user_budget, budget.remove_subcategory, key)
    return remove


def add_subcategory(user, user_budget, name, category):
    return changed(user, user_budget, budget.add_subcategory, name, category)


# ---------- Event handlers: goals ----------

def dollars_text(cents):
    """Pre-fill for an amount box: "$1,800", or blank when not set."""
    return budget.format_money(cents) if cents else ""


def money_box(label, placeholder="$0.00", **options):
    """A text box that shows $ and commas as you type (design.MONEY_INPUT_SCRIPT)."""
    return gr.Textbox(label=label, placeholder=placeholder, elem_classes="sb-money-input", min_width=0, **options)


def target_texts(targets):
    """Pre-fill for the four target boxes: "50", "30", "10", "10", or blanks when off."""
    complete = set(targets) == set(budget.CATEGORIES)
    return [str(targets[category]) if complete else "" for category in budget.CATEGORIES]


def parse_field(label, text):
    try:
        return budget.parse_optional_amount(text)
    except ValueError as error:
        raise ValueError(f"{label}: {error}")


def plan_line_html(limit_text, *percent_texts):
    """Under the Goals boxes: what each typed % comes to in dollars of the monthly limit."""
    amounts = budget.plan_amounts(limit_text, percent_texts)
    parts = [f"{budget.CATEGORY_NAMES[category]} {budget.format_money(cents)}" for category, cents in amounts.items()]
    return f'<div class="sb-note">{" · ".join(parts)}</div>' if parts else ""


def set_plan(updated, limit_text, targets):
    """Store the monthly limit and the targets, and remember a plan was chosen so sign-in stops offering presets."""
    # A fresh dict, so the Needs and Wants limits from before they followed the targets don't linger.
    updated["settings"]["limits"] = {"overall": parse_field(MONTHLY_LIMIT, limit_text)}
    updated["settings"]["targets"] = targets
    updated["settings"]["plan_chosen"] = True


def plan_outputs(updated):
    """What a saved plan refreshes: the budget, the Goals limit and % boxes, and the auto-filled box (none now)."""
    settings = updated["settings"]
    return updated, dollars_text(settings["limits"]["overall"]), *target_texts(settings["targets"]), None


def save_plan(user, user_budget, limit_text, needs, wants, investing, savings):
    def apply(updated):
        set_plan(updated, limit_text, budget.parse_targets(needs, wants, investing, savings))
    updated = changed(user, user_budget, apply)
    gr.Info("Plan saved" if updated["settings"]["targets"] else "Plan saved, targets off", title="Saved")
    return plan_outputs(updated)


def make_use_preset(preset):
    """The chooser's "Use this": save the preset's targets with the limit typed there, and close."""
    def use(user, user_budget, limit_text):
        updated = changed(user, user_budget, set_plan, limit_text, dict(preset["targets"]))
        gr.Info(f"{preset['name']} plan saved", title="Saved")
        return *plan_outputs(updated), gr.update(visible=False)
    return use


def set_own_plan(user, user_budget, limit_text):
    """The chooser's "Set my own": keep the limit typed there and the current targets, and close."""
    updated = changed(user, user_budget, lambda changing: set_plan(changing, limit_text, changing["settings"]["targets"]))
    return *plan_outputs(updated), gr.update(visible=False)


def save_goal(user, user_budget, goal):
    updated = changed(user, user_budget, lambda changing: changing["settings"].update(savings_goal=parse_field("Savings goal", goal)))
    gr.Info("Savings goal saved", title="Saved")
    return updated


def preset_html(preset):
    """A preset's card in the chooser: its name, its four shares, why, and the research behind it."""
    split = " · ".join(f"{budget.CATEGORY_NAMES[category]} {preset['targets'][category]}%" for category in budget.CATEGORIES)
    return (
        f'<div class="sb-title">{escape(preset["name"])}</div><div class="sb-preset-split">{split}</div>'
        f'<div class="sb-note">{escape(preset["why"])}</div><div class="sb-source">{escape(preset["source"])}</div>'
    )


def make_complete(edited):
    """Typing in one target box: when three are filled, the fourth makes 100 and keeps doing so."""
    def complete(needs, wants, investing, savings, auto_key):
        typed = [needs, wants, investing, savings]
        filled, auto_key = budget.complete_targets(typed, auto_key, edited)
        # Only a box whose value really changes is updated, so no other box's change event fires for nothing.
        others = [new if new != old else gr.skip() for category, old, new in zip(budget.CATEGORIES, typed, filled) if category != edited]
        return *others, auto_key
    return complete


# ---------- Event handlers: history ----------

def zoom_choices_update(user_budget, pick):
    """The hidden zoom radio follows the subcategories; a pick that no longer exists goes back to everything."""
    if not user_budget:
        return gr.skip()
    choices = charts.zoom_choices(user_budget)
    return gr.update(choices=choices, value=pick if pick in {value for _, value in choices} else "all")


def history_view(user_budget, range_choice, pick):
    """The History tab for a range and a zoom: tiles, chart, biggest items, and whether to offer samples."""
    if not user_budget:
        return "", "", "", gr.update(visible=False), gr.update(visible=False)
    end_key, count = budget.history_range(user_budget, range_choice, budget.month_key())
    rows = budget.month_history(user_budget, end_key, count)
    valid = {value for _, value in charts.zoom_choices(user_budget)}
    view = charts.chart_view(user_budget, rows, pick if pick in valid else "all")
    items = charts.top_items_html(charts.top_items(user_budget, rows, view), view) if view["level"] != "all" else ""
    offer_samples = budget.months_with_data(user_budget) < SAMPLE_OFFER_BELOW
    return (
        charts.stat_tiles_html(rows, view), charts.history_chart(rows, view), items,
        gr.update(visible=bool(items)), gr.update(visible=offer_samples),
    )


def fill_samples(user, user_budget):
    updated = copy.deepcopy(user_budget)
    filled = budget.fill_sample_months(updated, budget.month_key())
    storage.save_budget(user, updated)
    gr.Info(f"Added {filled} sample months" if filled else "Those months already have data", title="Samples")
    return updated


# ---------- Event handlers: account ----------

def sign_in(name_text, password):
    """A known name needs its password; a new name, or a budget saved before passwords, takes this one."""
    try:
        key, user_budget = storage.sign_in(name_text, password)
    except ValueError as error:
        raise fail(str(error))
    settings = user_budget["settings"]
    limit = dollars_text(settings["limits"].get("overall", 0))
    # Offered once: a new account picks a plan or sets its own, and either way it isn't asked again.
    ask_for_plan = not settings.get("plan_chosen") and not settings.get("targets")
    return (
        key, user_budget, budget.month_key(), None,
        gr.update(visible=False), gr.update(visible=True), gr.Tabs(selected="budget", visible=True),
        limit, dollars_text(settings["savings_goal"]),
        *target_texts(settings.get("targets", {})),
        "", gr.update(visible=ask_for_plan), limit,
    )


def sign_out():
    return (
        None, None, budget.month_key(), None, gr.update(visible=True), gr.update(visible=False),
        "", False, gr.update(visible=False), gr.update(visible=False),
    )


def toggle_menu(menu_open):
    return not menu_open, gr.update(visible=not menu_open)


def open_profile(user_budget):
    """The profile page takes the tabs' place until Back."""
    account = user_budget["account"]
    return False, gr.update(visible=False), gr.update(visible=True), gr.update(visible=False), account["name"], account["email"]


def save_profile(user, user_budget, name_text, email_text):
    """Change the sign-in name (moving the saved file) and the email."""
    try:
        name = storage.clean_name(name_text)
        new_key = storage.file_key(name)
        email = storage.clean_email(email_text)
        if new_key != user:
            storage.rename_budget(user, new_key)
    except ValueError as error:
        raise fail(str(error))
    updated = copy.deepcopy(user_budget)
    updated["account"].update(name=name, email=email)
    storage.save_budget(new_key, updated)
    gr.Info("Profile saved", title="Saved")
    return new_key, updated


def change_password(user, user_budget, current, new, confirm):
    if not storage.check_password(user_budget["account"], current):
        raise fail("Current password is wrong")
    if new != confirm:
        raise fail("The new passwords don't match")
    try:
        hashed = storage.new_password(new)
    except ValueError as error:
        raise fail(str(error))
    updated = copy.deepcopy(user_budget)
    updated["account"].update(hashed)
    storage.save_budget(user, updated)
    gr.Info("Password changed", title="Saved")
    return updated, "", "", ""


def delete_data(user):
    storage.delete_budget(user)
    gr.Info("Your data is deleted", title="Deleted")
    return (*sign_out(), gr.update(visible=False))


# ---------- Layout ----------

with gr.Blocks(title="Sliced Bread") as demo:
    user_state = gr.State(None)  # file key of the signed-in name, "mark_g"
    budget_state = gr.State(None)  # that name's whole budget
    month_state = gr.State(budget.month_key())  # month on screen, "2026-10"
    open_state = gr.State(None)  # ("add", section) or ("edit", section, index): the one open form
    auto_target_state = gr.State(None)  # the target box filled in to make 100, until the user types in it
    menu_state = gr.State(False)  # whether the avatar's menu is open
    subs_open_state = gr.State(False)  # whether the subcategory pop-up is open

    # Sign in: a new name makes a new budget with that password
    with gr.Column(elem_classes="sb-card sb-auth") as auth_view:
        html(
            f'<div class="sb-auth-top">{design.duck_full(200)}<div class="sb-wordmark">Sliced Bread</div>'
            '<div class="sb-note">Know what is left to invest each month</div>'
            '<span class="sb-chip">New here? Pick a name and a password</span></div>'
        )
        login_username = gr.Textbox(label="Name", placeholder="Your name", html_attributes=gr.InputHTMLAttributes(autocomplete="username"))
        login_password = gr.Textbox(
            label="Password", type="password", placeholder=f"At least {storage.MIN_PASSWORD_LENGTH} characters",
            html_attributes=gr.InputHTMLAttributes(autocomplete="current-password"),
        )
        login_button = gr.Button("Sign in", variant="primary", elem_classes="sb-full")

    with gr.Column(visible=False, elem_classes="sb-app") as app_view:
        # Header: logo left; the person's name and an avatar that opens a small menu, right.
        with gr.Row(elem_classes="sb-header"):
            html(f'<div class="sb-brand">{design.duck_mark(36)}<span class="sb-wordmark">Sliced Bread</span></div>', elem_classes="sb-grow")
            person_name = html("", elem_classes="sb-person")
            avatar_button = gr.Button("", elem_classes="sb-avatar-button", min_width=0, scale=0)
        with gr.Column(visible=False, elem_classes="sb-card sb-menu") as person_menu:
            edit_profile_button = gr.Button("Edit profile", elem_classes="sb-menu-item")
            sign_out_button = gr.Button("Sign out", elem_classes="sb-menu-item")

        with gr.Tabs() as app_tabs:
            with gr.Tab("Budget", id="budget"):
                # The hero card: month switcher on top, then the big number, the bar and the duck.
                with gr.Column(elem_classes="sb-card"):
                    with gr.Row(elem_classes="sb-hero-top"):
                        previous_month = gr.Button("‹", elem_classes="sb-icon sb-nav", min_width=0, scale=0)
                        month_title = html("", elem_classes="sb-grow")
                        next_month = gr.Button("›", elem_classes="sb-icon sb-nav", min_width=0, scale=0)

                    @gr.render(inputs=[budget_state, month_state], triggers=[budget_state.change, month_state.change])
                    def render_hero(user_budget, key):
                        if user_budget:
                            html(hero_html(user_budget, key))

                @gr.render(inputs=[budget_state, month_state, open_state], triggers=[budget_state.change, month_state.change, open_state.change])
                def render_budget(user_budget, key, open_section):
                    if not user_budget:
                        return
                    month = budget.get_month(user_budget, key)
                    opened = tuple(open_section) if open_section else None
                    split = split_html(user_budget, month)
                    if split:
                        with gr.Column(elem_classes="sb-card"):
                            html(split)

                    for section, section_name, category in sections(user_budget):
                        is_open = opened == ("add", section)
                        with gr.Column(elem_classes="sb-card"):
                            with gr.Row(elem_classes="sb-row"):
                                html(section_head_html(user_budget, month, section, section_name, category), elem_classes="sb-grow")
                                if not is_open:  # an open form closes with its own Done
                                    plus = gr.Button("+ Add", variant="primary", elem_classes="sb-add-button", min_width=0, scale=0)
                                    plus.click(make_open(("add", section)), outputs=open_state)

                            if is_open:
                                with gr.Column(elem_classes="sb-panel"):
                                    # Start in the tier last used this month, so several lines go in one after another.
                                    last_tier = month[section][-1]["tier"] if section == budget.TIERED_SECTION and month[section] else ""
                                    tier_input = tier_box(user_budget, last_tier) if section == budget.TIERED_SECTION else None
                                    with gr.Row(elem_classes="sb-row"):
                                        name_box = gr.Textbox(
                                            label="Name", placeholder=NAME_PLACEHOLDERS.get(section, ""),
                                            autofocus=tier_input is None or bool(last_tier), scale=3, min_width=0,
                                        )
                                        amount_box = money_box("Amount", scale=2)
                                    with gr.Row(elem_classes="sb-row"):
                                        add_button = gr.Button("Add", variant="primary")
                                        cancel_button = gr.Button("Done")
                                add_inputs = [user_state, budget_state, month_state, name_box, amount_box] + ([tier_input] if tier_input else [])
                                for trigger in (add_button.click, amount_box.submit, name_box.submit):
                                    trigger(make_add(section), inputs=add_inputs, outputs=[budget_state, name_box, amount_box])
                                cancel_button.click(lambda: None, outputs=open_state)

                            items = month[section]
                            if not items and not is_open:
                                html(f'<div class="sb-note">{EMPTY_STATES.get(section, CUSTOM_EMPTY_STATE)}</div>')
                            # Investment lines show under their tier; each keeps its real position for edit and delete.
                            groups = [(None, list(enumerate(items)))]
                            if section == budget.TIERED_SECTION:
                                names = list(dict.fromkeys(item["tier"] for item in items))
                                groups = [(tier, [(i, item) for i, item in enumerate(items) if item["tier"] == tier]) for tier in names]
                            for tier, lines in groups:
                                if tier:
                                    html(tier_head_html(tier, sum(item["amount"] for _, item in lines)))
                                for index, item in lines:
                                    if opened == ("edit", section, index):
                                        with gr.Column(elem_classes="sb-panel"):
                                            edit_tier = tier_box(user_budget, item["tier"]) if tier else None
                                            with gr.Row(elem_classes="sb-row"):
                                                edit_name = gr.Textbox(label="Name", value=item["name"], autofocus=True, interactive=True, scale=3, min_width=0)
                                                edit_amount = money_box("Amount", value=dollars_text(item["amount"]), interactive=True, scale=2)
                                            with gr.Row(elem_classes="sb-row"):
                                                save_button = gr.Button("Save", variant="primary")
                                                cancel_edit_button = gr.Button("Cancel")
                                        edit_inputs = [user_state, budget_state, month_state, edit_name, edit_amount] + ([edit_tier] if edit_tier else [])
                                        for trigger in (save_button.click, edit_name.submit, edit_amount.submit):
                                            trigger(make_update(section, index), inputs=edit_inputs, outputs=[budget_state, open_state])
                                        cancel_edit_button.click(lambda: None, outputs=open_state)
                                        continue
                                    with gr.Row(elem_classes="sb-row sb-item"):
                                        html(design.line_html(item["name"], item["amount"]), elem_classes="sb-grow")
                                        edit = gr.Button("✎", elem_classes="sb-icon", min_width=0, scale=0)
                                        edit.click(make_open(("edit", section, index)), outputs=open_state)
                                        remove = gr.Button("✕", elem_classes="sb-icon", min_width=0, scale=0)
                                        remove.click(make_remove(section, index), inputs=[user_state, budget_state, month_state, open_state], outputs=[budget_state, open_state])

                    with gr.Column(elem_classes="sb-card sb-card-accent"):
                        html(savings_html(user_budget, key))

            with gr.Tab("History", id="history"):
                range_pick = gr.Radio(RANGE_CHOICES, value=RANGE_CHOICES[0], show_label=False, container=False, elem_classes="sb-pills")
                # Hidden: the chart's coloured pieces, legend and back link set it (design.HEAD's click script).
                zoom_pick = gr.Radio([(charts.ALL_CHOICE, "all")], value="all", show_label=False, container=False, elem_classes="sb-hidden sb-category-pick")
                with gr.Column(elem_classes="sb-card", visible=False) as sample_card:
                    html(
                        f'<div class="sb-duck-says" style="margin:0;padding:0;border:0">{design.duck_mark(44)}'
                        '<div class="sb-bubble">Only a month or two so far. Want sample months to see how this looks?</div></div>'
                    )
                    sample_button = gr.Button("Fill 5 sample months", variant="primary")
                history_tiles = html("")
                with gr.Column(elem_classes="sb-card"):
                    history_chart = html("")
                with gr.Column(elem_classes="sb-card", visible=False) as items_card:
                    history_items = html("")

            with gr.Tab("Goals", id="goals"):
                # The monthly limit times each % is that category's limit (Needs, Wants) or goal (Investing, Savings).
                with gr.Column(elem_classes="sb-card"):
                    html('<div class="sb-title">Your plan</div>')
                    limit_overall = money_box(MONTHLY_LIMIT, placeholder="No limit")
                    with gr.Row():
                        target_needs = gr.Textbox(label="Needs %", placeholder="Off", min_width=0)
                        target_wants = gr.Textbox(label="Wants %", placeholder="Off", min_width=0)
                    with gr.Row():
                        target_investing = gr.Textbox(label="Investing %", placeholder="Off", min_width=0)
                        target_savings = gr.Textbox(label="Savings %", placeholder="Off", min_width=0)
                    plan_line = html("")
                    with gr.Row():
                        choose_preset_button = gr.Button("Choose a preset")
                        save_plan_button = gr.Button("Save plan", variant="primary")
                    edit_subs_button = gr.Button("Edit subcategories")

                with gr.Column(elem_classes="sb-card"):
                    html('<div class="sb-title">Savings goal</div>')
                    savings_goal = money_box("Savings goal", placeholder="No goal")
                    save_goal_button = gr.Button("Save goal", variant="primary")

        # The subcategory editor: a pop-up over the page from Goals' "Edit subcategories". Renames and moves wait for
        # Confirm; ✕ and Add subcategory save straight away.
        with gr.Column(visible=False, elem_classes="sb-modal") as subs_modal:
            with gr.Column(elem_classes="sb-card sb-modal-card"):
                html('<div class="sb-title">Subcategories</div>')

                @gr.render(inputs=[budget_state, subs_open_state], triggers=[budget_state.change, subs_open_state.change])
                def render_subcategories(user_budget, subs_open):
                    if not (user_budget and subs_open):
                        return
                    keys, name_boxes, move_boxes = [], [], []
                    for category in budget.CATEGORIES:
                        html(f'<div class="sb-group-head">{design.marker(category)}</div>')
                        for sub in budget.subcategories_in(user_budget, category):
                            with gr.Row(elem_classes="sb-row sb-sub-row"):
                                name_boxes.append(gr.Textbox(value=sub["name"], show_label=False, interactive=True, max_lines=1, scale=1, min_width=0))
                                move_boxes.append(gr.Dropdown(CATEGORY_OPTIONS, value=category, show_label=False, interactive=True, scale=0, min_width=CATEGORY_PICKER_WIDTH))
                                remove_button = gr.Button("✕", elem_classes="sb-icon", min_width=0, scale=0)
                                remove_button.click(make_remove_subcategory(sub["key"]), inputs=[user_state, budget_state], outputs=budget_state)
                            if sub["key"] in budget.PRESET_KEYS:
                                html('<span class="sb-preset-tag">Preset</span>')
                            keys.append(sub["key"])

                    with gr.Row(elem_classes="sb-row sb-add-sub"):
                        new_sub_name = gr.Textbox(label="New subcategory", placeholder="Pets", interactive=True, max_lines=1, scale=1, min_width=0)
                        new_sub_category = gr.Dropdown(CATEGORY_OPTIONS, value="wants", label="Counts toward", interactive=True, scale=0, min_width=CATEGORY_PICKER_WIDTH)
                    add_sub_button = gr.Button("Add subcategory")
                    for trigger in (add_sub_button.click, new_sub_name.submit):
                        trigger(add_subcategory, inputs=[user_state, budget_state, new_sub_name, new_sub_category], outputs=budget_state)

                    with gr.Row(elem_classes="sb-row"):
                        confirm_button = gr.Button("Confirm", variant="primary")
                        cancel_subs_button = gr.Button("Cancel")
                    confirm_button.click(make_confirm(keys), inputs=[user_state, budget_state, *name_boxes, *move_boxes], outputs=[budget_state, subs_open_state])
                    cancel_subs_button.click(lambda: False, outputs=subs_open_state)

        # The plan chooser: offered once after a new account's first sign-in, then from Goals' "Choose a preset".
        with gr.Column(visible=False, elem_classes="sb-modal") as plan_modal:
            with gr.Column(elem_classes="sb-card sb-modal-card"):
                html('<div class="sb-title">Pick a plan</div><div class="sb-note">You can change any number later on Goals.</div>')
                plan_limit = money_box(MONTHLY_LIMIT, placeholder="No limit")
                preset_buttons = []
                for preset in budget.PLAN_PRESETS:
                    with gr.Column(elem_classes="sb-panel"):
                        html(preset_html(preset))
                        preset_buttons.append((preset, gr.Button("Use this", variant="primary")))
                set_own_button = gr.Button("Set my own")

        # Opened from the avatar menu's "Edit profile" in place of the tabs. Not a tab: Gradio 6.31 folds a fourth tab
        # into a "more" menu even when it fits.
        with gr.Column(visible=False) as profile_view:
            profile_back_button = gr.Button("← Back", elem_classes="sb-back-button")
            with gr.Column(elem_classes="sb-card"):
                html('<div class="sb-title">Profile</div>')
                profile_name = gr.Textbox(label="Name", html_attributes=gr.InputHTMLAttributes(autocomplete="username"))
                profile_email = gr.Textbox(label="Email", placeholder="name@example.com", html_attributes=gr.InputHTMLAttributes(autocomplete="email"))
                save_profile_button = gr.Button("Save profile", variant="primary")

            with gr.Column(elem_classes="sb-card"):
                html('<div class="sb-title">Change password</div>')
                current_password = gr.Textbox(label="Current password", type="password", html_attributes=gr.InputHTMLAttributes(autocomplete="current-password"))
                new_password = gr.Textbox(label="New password", type="password", placeholder=f"At least {storage.MIN_PASSWORD_LENGTH} characters", html_attributes=gr.InputHTMLAttributes(autocomplete="new-password"))
                confirm_password = gr.Textbox(label="New password again", type="password", html_attributes=gr.InputHTMLAttributes(autocomplete="new-password"))
                change_password_button = gr.Button("Change password", variant="primary")

            with gr.Column(elem_classes="sb-card sb-danger"):
                delete_button = gr.Button("Delete my data", variant="stop")
                with gr.Column(visible=False) as delete_confirm:
                    html('<div class="sb-note">This removes every month saved under your name. It cannot be undone.</div>')
                    with gr.Row():
                        confirm_delete_button = gr.Button("Delete for good", variant="stop")
                        keep_button = gr.Button("Keep my data")

    # ---------- Wiring ----------

    target_boxes = [target_needs, target_wants, target_investing, target_savings]
    session_outputs = [user_state, budget_state, month_state, open_state, auth_view, app_view]
    signed_in_outputs = session_outputs + [app_tabs, limit_overall, savings_goal, *target_boxes, login_password, plan_modal, plan_limit]
    signed_out_outputs = session_outputs + [login_password, menu_state, person_menu, profile_view]

    for trigger in (login_button.click, login_username.submit, login_password.submit):
        trigger(sign_in, inputs=[login_username, login_password], outputs=signed_in_outputs)

    # The header follows the budget's name.
    budget_state.change(lambda user_budget: (person_html(user_budget), initial(user_budget)), inputs=budget_state, outputs=[person_name, avatar_button])

    month_state.change(month_title_html, inputs=month_state, outputs=month_title)
    demo.load(month_title_html, inputs=month_state, outputs=month_title)
    previous_month.click(lambda key: (budget.shift_month(key, -1), None), inputs=month_state, outputs=[month_state, open_state])
    next_month.click(lambda key: (budget.shift_month(key, 1), None), inputs=month_state, outputs=[month_state, open_state])

    for target, box in zip(budget.CATEGORIES, target_boxes):
        others = [other for other_target, other in zip(budget.CATEGORIES, target_boxes) if other_target != target]
        # .change, not .input: Gradio 6.31 never sent this textbox's input event. complete_targets tells its own
        # update of the filled-in box apart from the user typing over it.
        box.change(
            make_complete(target), inputs=[*target_boxes, auto_target_state], outputs=[*others, auto_target_state],
            trigger_mode="always_last", show_progress="hidden",
        )
    for box in (limit_overall, *target_boxes):
        box.change(plan_line_html, inputs=[limit_overall, *target_boxes], outputs=plan_line, trigger_mode="always_last", show_progress="hidden")
    plan_saved_outputs = [budget_state, limit_overall, *target_boxes, auto_target_state]
    save_plan_button.click(save_plan, inputs=[user_state, budget_state, limit_overall, *target_boxes], outputs=plan_saved_outputs)
    choose_preset_button.click(lambda limit: (gr.update(visible=True), limit), inputs=limit_overall, outputs=[plan_modal, plan_limit])
    for preset, button in preset_buttons:
        button.click(make_use_preset(preset), inputs=[user_state, budget_state, plan_limit], outputs=plan_saved_outputs + [plan_modal])
    set_own_button.click(set_own_plan, inputs=[user_state, budget_state, plan_limit], outputs=plan_saved_outputs + [plan_modal])
    edit_subs_button.click(lambda: True, outputs=subs_open_state)
    subs_open_state.change(lambda subs_open: gr.update(visible=subs_open), inputs=subs_open_state, outputs=subs_modal)
    save_goal_button.click(save_goal, inputs=[user_state, budget_state, savings_goal], outputs=budget_state)

    history_outputs = [history_tiles, history_chart, history_items, items_card, sample_card]
    history_inputs = [budget_state, range_pick, zoom_pick]
    budget_state.change(zoom_choices_update, inputs=[budget_state, zoom_pick], outputs=zoom_pick).then(
        history_view, inputs=history_inputs, outputs=history_outputs,
    )
    for trigger in (range_pick.change, zoom_pick.change):
        trigger(history_view, inputs=history_inputs, outputs=history_outputs)
    sample_button.click(fill_samples, inputs=[user_state, budget_state], outputs=budget_state)

    avatar_button.click(toggle_menu, inputs=menu_state, outputs=[menu_state, person_menu])
    edit_profile_button.click(open_profile, inputs=budget_state, outputs=[menu_state, person_menu, profile_view, app_tabs, profile_name, profile_email])
    profile_back_button.click(lambda: (gr.update(visible=False), gr.update(visible=True)), outputs=[profile_view, app_tabs])
    sign_out_button.click(sign_out, outputs=signed_out_outputs)
    save_profile_button.click(save_profile, inputs=[user_state, budget_state, profile_name, profile_email], outputs=[user_state, budget_state])
    change_password_button.click(
        change_password, inputs=[user_state, budget_state, current_password, new_password, confirm_password],
        outputs=[budget_state, current_password, new_password, confirm_password],
    )
    delete_button.click(lambda: gr.update(visible=True), outputs=delete_confirm)
    keep_button.click(lambda: gr.update(visible=False), outputs=delete_confirm)
    confirm_delete_button.click(delete_data, inputs=user_state, outputs=signed_out_outputs + [delete_confirm])


if __name__ == "__main__":
    demo.launch(theme=design.build_theme(), css=design.CSS, head=design.HEAD, share="--share" in sys.argv)  # Colab needs the public link


## `test_budget.py`

Automated tests for budget.py and storage.py.

In [ ]:
%%writefile test_budget.py
"""Tests for budget.py, storage.py and the History zoom helpers. Run with: python -m pytest"""

from datetime import date

import pytest

import budget
import charts
import design
import storage

TARGETS_45_30_20_5 = {"needs": 45, "wants": 30, "investing": 20, "savings": 5}  # what the meter tests were written for


# ---------- budget.py: input validation ----------

@pytest.mark.parametrize("typed, cents", [
    ("12", 1200), ("12.5", 1250), ("12.50", 1250), ("$1,200", 120000), (" 3 ", 300), (".99", 99), (7, 700),
])
def test_parse_amount_accepts_normal_input(typed, cents):
    assert budget.parse_amount(typed) == cents


@pytest.mark.parametrize("typed, message", [
    ("", "Enter an amount"), (None, "Enter an amount"), ("abc", "must be a number"), ("1.2.3", "must be a number"),
    ("-5", "negative"), ("0", "more than"), ("0.00", "more than"), ("1.234", "2 decimal"),
    ("10000001", "too large"), (".", "must be a number"),
])
def test_parse_amount_rejects_bad_input(typed, message):
    with pytest.raises(ValueError, match=message):
        budget.parse_amount(typed)


def test_optional_amount_treats_blank_and_zero_as_unset():
    assert budget.parse_optional_amount("") == 0
    assert budget.parse_optional_amount("0.00") == 0
    assert budget.parse_optional_amount("1,800") == 180000


def test_clean_name_trims_and_rejects_blank_or_long():
    assert budget.clean_name("  Rent   and  water ") == "Rent and water"
    with pytest.raises(ValueError, match="Enter a name"):
        budget.clean_name("   ")
    with pytest.raises(ValueError, match="40 characters"):
        budget.clean_name("x" * 41)


@pytest.mark.parametrize("cents, shown", [(120000, "$1,200"), (1250, "$12.50"), (-8500, "-$85"), (0, "$0"), (5, "$0.05")])
def test_format_money(cents, shown):
    assert budget.format_money(cents) == shown


# ---------- budget.py: calculations ----------

def make_budget():
    user_budget = budget.empty_budget()
    for section, name, amount in [
        ("income", "Salary", "3000"), ("income", "Freelance", "200"),
        ("necessities", "Rent", "1200"), ("necessities", "Groceries", "380.50"),
        ("discretionary", "Movies", "40"), ("travel", "Flights", "300"),
    ]:
        budget.add_item(user_budget, "2026-10", section, name, amount)
    return user_budget


def test_totals_and_leftover():
    user_budget = make_budget()
    month = budget.get_month(user_budget, "2026-10")
    assert budget.section_total(month, "income") == 320000
    assert budget.total_expenses(user_budget, month) == 120000 + 38050 + 4000 + 30000
    assert budget.leftover(user_budget, month) == 320000 - 192050


def test_leftover_goes_negative_when_overspent():
    user_budget = budget.empty_budget()
    budget.add_item(user_budget, "2026-10", "income", "Job", "100")
    budget.add_item(user_budget, "2026-10", "necessities", "Rent", "185")
    assert budget.leftover(user_budget, budget.get_month(user_budget, "2026-10")) == -8500


def test_empty_month_has_zero_everything():
    user_budget = budget.empty_budget()
    month = budget.get_month(user_budget, "2030-01")
    assert budget.leftover(user_budget, month) == 0 and budget.total_expenses(user_budget, month) == 0


def test_remove_item():
    user_budget = make_budget()
    budget.remove_item(user_budget, "2026-10", "income", 1)
    assert budget.section_total(budget.get_month(user_budget, "2026-10"), "income") == 300000
    with pytest.raises(IndexError):
        budget.remove_item(user_budget, "2026-10", "income", 5)


def test_add_item_rejects_unknown_section_and_bad_amount_without_changing_anything():
    user_budget = budget.empty_budget()
    with pytest.raises(ValueError):
        budget.add_item(user_budget, "2026-10", "investments", "Stocks", "10")  # no tier
    with pytest.raises(ValueError):
        budget.add_item(user_budget, "2026-10", "income", "Job", "lots")
    assert user_budget["months"] == {}


@pytest.mark.parametrize("spent, limit, state, label", [
    (100, 0, "none", ""),
    (50000, 100000, "good", "$500 left"),
    (90000, 100000, "warn", "Near limit, $100 left"),
    (100000, 100000, "warn", "Near limit, $0 left"),
    (104500, 100000, "over", "Over by $45"),
])
def test_meter_states(spent, limit, state, label):
    result = budget.meter(spent, limit)
    assert result["state"] == state and result["label"] == label
    assert 0 <= result["fraction"] <= 1


def test_savings_progress_ignores_overspent_months():
    user_budget = make_budget()
    budget.add_item(user_budget, "2026-11", "income", "Job", "100")
    budget.add_item(user_budget, "2026-11", "travel", "Trip", "500")
    assert budget.savings_progress(user_budget) == 320000 - 192050


@pytest.mark.parametrize("key, steps, result", [("2026-10", 1, "2026-11"), ("2026-12", 1, "2027-01"), ("2026-01", -1, "2025-12")])
def test_shift_month(key, steps, result):
    assert budget.shift_month(key, steps) == result


def test_month_key_and_label():
    assert budget.month_key(date(2026, 3, 9)) == "2026-03"
    assert budget.month_label("2026-10") == "October 2026"


# ---------- budget.py: editing a line ----------

def test_update_item_replaces_name_and_amount():
    user_budget = make_budget()
    budget.update_item(user_budget, "2026-10", "necessities", 0, "Rent and water", "1,250")
    month = budget.get_month(user_budget, "2026-10")
    assert month["necessities"][0] == {"name": "Rent and water", "amount": 125000}
    assert budget.section_total(month, "necessities") == 125000 + 38050


def test_update_item_rejects_bad_input_without_changing_anything():
    user_budget = make_budget()
    with pytest.raises(ValueError):
        budget.update_item(user_budget, "2026-10", "necessities", 0, "Rent", "-5")
    with pytest.raises(IndexError):
        budget.update_item(user_budget, "2026-10", "necessities", 9, "Rent", "5")
    assert budget.get_month(user_budget, "2026-10")["necessities"][0] == {"name": "Rent", "amount": 120000}


# ---------- budget.py: percentage targets ----------

@pytest.mark.parametrize("typed, targets", [
    (("50", "30", "10", "10"), {"needs": 50, "wants": 30, "investing": 10, "savings": 10}),
    (("60%", " 20 ", "15", "5"), {"needs": 60, "wants": 20, "investing": 15, "savings": 5}),
    (("", "", "", ""), {}),
    ((None, None, None, None), {}),
])
def test_parse_targets_accepts(typed, targets):
    assert budget.parse_targets(*typed) == targets


@pytest.mark.parametrize("typed, message", [
    (("50", "30", "10", "0"), r"add up to 100% \(now 90%\)"),
    (("50", "", "10", "10"), "Wants must be a whole number"),
    (("50.5", "29.5", "10", "10"), "Needs must be a whole number"),
    (("150", "-30", "-10", "-10"), "Needs must be a whole number"),
    (("50", "30", "x", "10"), "Investing must be a whole number"),
])
def test_parse_targets_rejects(typed, message):
    with pytest.raises(ValueError, match=message):
        budget.parse_targets(*typed)


def test_target_meters_compare_spending_to_share_of_income():
    # Income $3,200; needs $1,580.50; wants $340; nothing invested; saved $1,279.50.
    user_budget = make_budget()
    meters = budget.target_meters(user_budget, TARGETS_45_30_20_5, budget.get_month(user_budget, "2026-10"))
    assert meters["needs"]["actual_percent"] == 49 and meters["needs"]["state"] == "over"
    assert meters["needs"]["label"] == "Over by $140.50"
    assert meters["wants"]["actual_percent"] == 11 and meters["wants"]["label"] == "$620 left"
    assert meters["investing"]["actual_percent"] == 0 and meters["investing"]["label"] == "$640 short"
    assert meters["savings"]["actual_percent"] == 40 and meters["savings"]["label"] == "On target"


def test_target_meters_flag_overspending_and_short_minimums():
    user_budget = budget.empty_budget()
    budget.add_item(user_budget, "2026-10", "income", "Job", "1000")
    budget.add_item(user_budget, "2026-10", "necessities", "Rent", "600")
    budget.add_item(user_budget, "2026-10", "travel", "Trip", "300")
    budget.add_item(user_budget, "2026-10", "investments", "Roth IRA", "60", "Retirement")
    meters = budget.target_meters(user_budget, TARGETS_45_30_20_5, budget.get_month(user_budget, "2026-10"))
    assert meters["needs"]["label"] == "Over by $150"
    assert meters["wants"]["state"] == "warn"
    assert meters["investing"]["state"] == "warn" and meters["investing"]["label"] == "$140 short"
    assert meters["savings"]["state"] == "warn" and meters["savings"]["label"] == "$10 short"


def test_target_meters_empty_without_targets_or_income():
    user_budget = make_budget()
    month = budget.get_month(user_budget, "2026-10")
    assert budget.target_meters(user_budget, {}, month) == {}
    assert budget.target_meters(user_budget, TARGETS_45_30_20_5, budget.get_month(user_budget, "2030-01")) == {}


def test_targets_saved_with_three_shares_count_as_off():
    old_style = {"needs": 50, "wants": 30, "savings": 20}
    user_budget = make_budget()
    assert budget.target_meters(user_budget, old_style, budget.get_month(user_budget, "2026-10")) == {}


# ---------- budget.py: investments and savings ----------

def test_investing_comes_out_before_savings():
    user_budget = make_budget()
    budget.add_item(user_budget, "2026-10", "investments", "Roth IRA", "500", "Retirement")
    month = budget.get_month(user_budget, "2026-10")
    assert budget.total_invested(user_budget, month) == 50000
    assert budget.total_expenses(user_budget, month) == 192050  # spending only, so limits ignore investing
    assert budget.leftover(user_budget, month) == 320000 - 192050 - 50000


def test_investment_lines_need_a_tier_and_other_lines_ignore_it():
    user_budget = budget.empty_budget()
    with pytest.raises(ValueError, match="Enter a tier"):
        budget.add_item(user_budget, "2026-10", "investments", "Roth IRA", "100", "  ")
    item = budget.add_item(user_budget, "2026-10", "investments", "Roth IRA", "100", "  Retirement   fund ")
    assert item == {"name": "Roth IRA", "amount": 10000, "tier": "Retirement fund"}
    assert budget.add_item(user_budget, "2026-10", "income", "Job", "100", "Ignored") == {"name": "Job", "amount": 10000}


def test_update_item_can_move_an_investment_to_another_tier():
    user_budget = budget.empty_budget()
    budget.add_item(user_budget, "2026-10", "investments", "Index fund", "100", "Retirement")
    budget.update_item(user_budget, "2026-10", "investments", 0, "Index fund", "150", "Brokerage")
    assert budget.get_month(user_budget, "2026-10")["investments"][0] == {"name": "Index fund", "amount": 15000, "tier": "Brokerage"}


def test_typing_a_tier_in_other_capitals_reuses_it():
    user_budget = budget.empty_budget()
    budget.add_item(user_budget, "2026-10", "investments", "Roth IRA", "100", "Retirement")
    assert budget.add_item(user_budget, "2026-11", "investments", "401k", "100", "retirement")["tier"] == "Retirement"
    assert budget.tiers(user_budget) == ["Retirement"]


def test_tiers_in_first_used_order():
    user_budget = budget.empty_budget()
    budget.add_item(user_budget, "2026-11", "investments", "Crypto", "50", "Fun money")
    budget.add_item(user_budget, "2026-10", "investments", "Roth IRA", "100", "Retirement")
    budget.add_item(user_budget, "2026-10", "investments", "Index fund", "100", "Brokerage")
    budget.add_item(user_budget, "2026-11", "investments", "Roth IRA", "100", "Retirement")
    assert budget.tiers(user_budget) == ["Retirement", "Brokerage", "Fun money"]


def test_budgets_saved_before_investments_existed_still_work():
    old_month = {"income": [{"name": "Job", "amount": 100000}], "necessities": [], "discretionary": [], "travel": []}
    user_budget = {**budget.empty_budget(), "months": {"2026-10": old_month}}
    assert budget.get_month(user_budget, "2026-10")["investments"] == []
    assert budget.leftover(user_budget, budget.get_month(user_budget, "2026-10")) == 100000
    budget.add_item(user_budget, "2026-10", "investments", "Roth IRA", "100", "Retirement")
    assert budget.total_invested(user_budget, budget.get_month(user_budget, "2026-10")) == 10000


# ---------- budget.py: subcategories ----------

def test_new_budgets_start_with_the_presets_in_their_categories():
    subs = budget.subcategories(budget.empty_budget())
    assert [(sub["key"], sub["category"]) for sub in subs] == [
        ("necessities", "needs"), ("discretionary", "wants"), ("travel", "wants"), ("investments", "investing"),
        ("emergency_fund", "savings"),
    ]


def test_budgets_saved_before_a_new_preset_get_it():
    user_budget = make_budget()
    pets = budget.add_subcategory(user_budget, "Pets", "wants")
    stored = user_budget["settings"]["subcategories"]
    stored.remove(next(sub for sub in stored if sub["key"] == "emergency_fund"))
    assert [sub["key"] for sub in budget.subcategories(user_budget)][-2:] == ["pets", "emergency_fund"]
    budget.rename_subcategory(user_budget, "emergency_fund", "Rainy day")
    assert stored[-1] == {"key": "emergency_fund", "name": "Rainy day", "category": "savings"} and pets in stored


def test_added_subcategory_counts_toward_its_category():
    user_budget = make_budget()
    pets = budget.add_subcategory(user_budget, "  Pets ", "wants")
    assert pets == {"key": "pets", "name": "Pets", "category": "wants"}
    budget.add_item(user_budget, "2026-10", "pets", "Vet", "60")
    month = budget.get_month(user_budget, "2026-10")
    assert budget.category_total(user_budget, month, "wants") == 4000 + 30000 + 6000
    assert budget.total_expenses(user_budget, month) == 192050 + 6000


def test_subcategory_names_must_be_unique_and_not_a_category():
    user_budget = budget.empty_budget()
    for name in ("travel", "Needs", "Income", "all spending"):
        with pytest.raises(ValueError, match="You already have"):
            budget.add_subcategory(user_budget, name, "wants")
    with pytest.raises(ValueError, match="Pick a category"):
        budget.add_subcategory(user_budget, "Pets", "fun")


def test_subcategory_keys_never_clash():
    user_budget = budget.empty_budget()
    budget.rename_subcategory(user_budget, "travel", "Trips")
    assert budget.add_subcategory(user_budget, "Travel", "wants")["key"] == "travel_2"
    assert budget.add_subcategory(user_budget, "!!!", "needs")["key"] == "subcategory"


def test_custom_subcategories_are_capped():
    user_budget = budget.empty_budget()
    for number in range(budget.MAX_CUSTOM_SUBCATEGORIES):
        budget.add_subcategory(user_budget, f"Extra {number}", "wants")
    with pytest.raises(ValueError, match="up to"):
        budget.add_subcategory(user_budget, "One more", "wants")


def test_moving_a_subcategory_moves_its_money_between_categories():
    user_budget = make_budget()
    budget.move_subcategory(user_budget, "travel", "needs")
    month = budget.get_month(user_budget, "2026-10")
    assert budget.category_total(user_budget, month, "needs") == 158050 + 30000
    assert budget.category_total(user_budget, month, "wants") == 4000
    assert [sub["key"] for sub in budget.subcategories_in(user_budget, "needs")] == ["necessities", "travel"]


def test_rename_keeps_the_lines():
    user_budget = make_budget()
    budget.rename_subcategory(user_budget, "travel", "Trips")
    assert budget.subcategories_in(user_budget, "wants")[1]["name"] == "Trips"
    assert budget.get_month(user_budget, "2026-10")["travel"][0]["name"] == "Flights"
    with pytest.raises(ValueError, match="You already have"):
        budget.rename_subcategory(user_budget, "travel", "discretionary")
    budget.rename_subcategory(user_budget, "travel", "TRIPS")  # its own name in other capitals is fine


def test_remove_refuses_subcategories_with_items():
    user_budget = make_budget()
    with pytest.raises(ValueError, match="has items in 1 month"):
        budget.remove_subcategory(user_budget, "travel")  # a preset, but it holds Flights
    budget.add_subcategory(user_budget, "Pets", "wants")
    budget.add_item(user_budget, "2026-10", "pets", "Vet", "60")
    with pytest.raises(ValueError, match="has items in 1 month. Delete them first"):
        budget.remove_subcategory(user_budget, "pets")
    budget.remove_item(user_budget, "2026-10", "pets", 0)
    budget.remove_subcategory(user_budget, "pets")
    assert "pets" not in budget.section_keys(user_budget)
    assert "pets" not in user_budget["months"]["2026-10"]


def test_a_removed_preset_stays_removed():
    user_budget = make_budget()
    budget.remove_subcategory(user_budget, "emergency_fund")
    assert "emergency_fund" not in budget.section_keys(user_budget)
    budget.rename_subcategory(user_budget, "travel", "Trips")  # writes the list out again; the preset must not return
    assert "emergency_fund" not in budget.section_keys(user_budget)


def test_removing_investments_leaves_tiers_and_sample_months_working():
    user_budget = budget.empty_budget()
    budget.remove_subcategory(user_budget, "investments")
    assert budget.tiers(user_budget) == []
    assert budget.fill_sample_months(user_budget, "2026-11") == 5
    assert budget.month_history(user_budget, "2026-10", 1)[0]["invested"] == 0
    assert budget.tiers(user_budget) == []


# ---------- budget.py: plan presets and the monthly limit ----------

def test_every_plan_preset_covers_all_four_categories_and_makes_100():
    for preset in budget.PLAN_PRESETS:
        assert set(preset["targets"]) == set(budget.CATEGORIES)
        assert sum(preset["targets"].values()) == budget.PERCENT_WHOLE
        assert budget.parse_targets(*(str(preset["targets"][category]) for category in budget.CATEGORIES)) == preset["targets"]


def test_category_limits_are_the_monthly_limit_times_each_target():
    settings = {"limits": {"overall": 300000}, "targets": {"needs": 50, "wants": 30, "investing": 15, "savings": 5}}
    assert budget.category_limits(settings) == {"needs": 150000, "wants": 90000, "investing": 45000, "savings": 15000}
    settings["limits"]["overall"] = 99999  # $999.99: shares round down to the cent
    assert budget.category_limits(settings)["savings"] == 4999


def test_category_limits_are_off_without_a_limit_or_targets():
    assert budget.category_limits({"limits": {"overall": 0}, "targets": TARGETS_45_30_20_5}) == {}
    assert budget.category_limits({"limits": {"overall": 300000}, "targets": {}}) == {}
    assert budget.category_limits(budget.empty_budget()["settings"]) == {}


def test_plan_amounts_follow_whatever_is_typed():
    assert budget.plan_amounts("$3,000", ["50", "30", "", "x"]) == {"needs": 150000, "wants": 90000}
    assert budget.plan_amounts("", ["50", "30", "15", "5"]) == {}
    assert budget.plan_amounts("abc", ["50", "30", "15", "5"]) == {}


def test_donut_has_a_segment_per_category_with_money():
    amounts = {"needs": 120000, "wants": 30000, "investing": 0, "savings": 50000}
    ring = design.donut_svg(amounts, 200000)
    assert ring.count("<circle") == 3 and "$2,000" in ring and "Needs $1,200 (60%)" in ring
    empty = design.donut_svg(dict.fromkeys(amounts, 0), 0)
    assert empty.count("<circle") == 1 and design.LINE in empty


def test_savings_subcategory_is_set_aside_not_left_over():
    user_budget = make_budget()
    budget.add_item(user_budget, "2026-10", "emergency_fund", "Transfer", "300")
    month = budget.get_month(user_budget, "2026-10")
    assert budget.leftover(user_budget, month) == 127950 - 30000
    assert budget.saved(user_budget, month) == 127950
    assert budget.savings_progress(user_budget) == 127950
    meters = budget.target_meters(user_budget, TARGETS_45_30_20_5, month)
    assert meters["savings"]["actual_percent"] == 40


@pytest.mark.parametrize("texts, auto_key, edited, result, new_auto", [
    (["50", "30", "10", ""], None, "investing", ["50", "30", "10", "10"], "savings"),  # three typed: fourth fills
    (["50", "", "10", ""], None, "investing", ["50", "", "10", ""], None),  # two blanks: wait
    (["60", "30", "10", "10"], "savings", "needs", ["60", "30", "10", "0"], "savings"),  # keeps in step
    (["70", "30", "10", "10"], "savings", "needs", ["70", "30", "10", ""], "savings"),  # over 100: blank, not negative
    (["50", "30", "10", "15"], "savings", "savings", ["50", "30", "10", "15"], None),  # typing in it hands it back
    (["50", "30", "10", "10"], "savings", "savings", ["50", "30", "10", "10"], "savings"),  # its own update
    (["50", "x", "10", ""], None, "wants", ["50", "x", "10", ""], None),    (["50", "30", "10", ""], None, "savings", ["50", "30", "10", ""], None),  # just cleared it: leave it alone
])
def test_complete_targets(texts, auto_key, edited, result, new_auto):
    assert budget.complete_targets(texts, auto_key, edited) == (result, new_auto)


# ---------- budget.py: history ----------

def test_month_history_is_oldest_first_with_empty_months_as_zeros():
    rows = budget.month_history(make_budget(), "2026-11", 3)
    assert [row["key"] for row in rows] == ["2026-09", "2026-10", "2026-11"]
    assert [row["label"] for row in rows] == ["Sep", "Oct", "Nov"]
    assert rows[0]["income"] == 0 and rows[0]["spent"] == 0 and rows[0]["left"] == 0
    october = rows[1]
    assert october["income"] == 320000 and october["spent"] == 192050 and october["left"] == 127950
    assert october["subcategories"] == {"necessities": 158050, "discretionary": 4000, "travel": 30000, "investments": 0,
                                     "emergency_fund": 0}
    assert october["categories"] == {"needs": 158050, "wants": 34000, "investing": 0, "savings": 0}
    assert october["invested"] == 0 and october["saved"] == 127950


def test_month_history_crosses_the_year_boundary():
    assert [row["key"] for row in budget.month_history(budget.empty_budget(), "2027-01", 3)] == ["2026-11", "2026-12", "2027-01"]


@pytest.mark.parametrize("choice, count", [("6 months", 6), ("12 months", 12)])
def test_history_range_fixed_windows_end_today(choice, count):
    assert budget.history_range(make_budget(), choice, "2026-11") == ("2026-11", count)


def test_history_range_all_starts_at_the_first_month_with_data():
    assert budget.history_range(make_budget(), "All", "2026-11") == ("2026-11", 2)
    assert budget.history_range(budget.empty_budget(), "All", "2026-11") == ("2026-11", 1)


def test_history_range_all_is_capped():
    user_budget = budget.empty_budget()
    budget.add_item(user_budget, "2020-01", "income", "Job", "100")
    assert budget.history_range(user_budget, "All", "2026-11") == ("2026-11", budget.MAX_HISTORY_MONTHS)


def test_top_items_largest_first_and_limited():
    user_budget = make_budget()
    budget.add_item(user_budget, "2026-09", "necessities", "Car repair", "900")
    items = budget.top_items(user_budget, ["2026-09", "2026-10"], "necessities", limit=2)
    assert items == [("Rent", "2026-10", 120000), ("Car repair", "2026-09", 90000)]


def test_section_lines_merges_same_label_and_keeps_empty_months():
    user_budget = budget.empty_budget()
    budget.add_item(user_budget, "2026-09", "necessities", "Rent", "1000")
    budget.add_item(user_budget, "2026-09", "necessities", "Rent", "200")
    budget.add_item(user_budget, "2026-09", "investments", "Index fund", "50", "Retirement")
    assert budget.section_lines(user_budget, ["2026-08", "2026-09"], "necessities") == {"2026-08": {}, "2026-09": {"Rent": 120000}}
    assert budget.section_lines(user_budget, ["2026-09"], "investments") == {"2026-09": {"Retirement: Index fund": 5000}}


def test_shades_run_darkest_to_lightest():
    colors = design.shades("#2A78D6", 4)
    assert len(colors) == 4 and all(len(color) == 7 and color.startswith("#") for color in colors)
    brightness = [sum(int(color[i:i + 2], 16) for i in (1, 3, 5)) for color in colors]
    assert brightness == sorted(brightness) and colors[0] != colors[-1]
    assert design.shades("#2A78D6", 1) == ["#2A78D6"]


def test_zoom_series_ranks_by_range_total_and_groups_the_rest():
    lines = {"2026-09": {f"Line {n}": n * 100 for n in range(1, 8)}, "2026-10": {"Line 1": 10_000}}
    names = [name for name, _ in charts.zoom_series(lines, "#2A78D6")]
    assert names == ["Line 1", "Line 7", "Line 6", "Line 5", "Line 4", charts.OTHER_LABEL]
    assert charts.month_amounts(lines["2026-09"], charts.zoom_series(lines, "#2A78D6"))[-1] == 300 + 200


def test_chart_views_drill_from_categories_to_subcategories_to_items():
    user_budget = make_budget()
    budget.add_item(user_budget, "2026-10", "emergency_fund", "Transfer", "100")
    rows = budget.month_history(user_budget, "2026-10", 2)
    everything = charts.chart_view(user_budget, rows, "all")
    assert [name for name, *_ in everything["series"]] == ["Needs", "Wants", "Investing", "Savings"]
    assert charts.view_totals(everything, rows) == [0, 192050 + 10000]
    wants = charts.chart_view(user_budget, rows, "category:wants")
    assert [(name, zoom) for name, _, zoom, _ in wants["series"]] == [("Discretionary", "Discretionary"), ("Travel", "Travel")]
    savings = charts.chart_view(user_budget, rows, "category:savings")
    assert savings["series"][-1][0] == charts.LEFT_OVER_LABEL and savings["series"][-1][3] == [0, 127950 - 10000]
    travel = charts.chart_view(user_budget, rows, "sub:travel")
    assert travel["back"] == "Wants" and travel["series"][0][0] == "Flights" and travel["series"][0][3] == [0, 30000]
    for view in (everything, wants, savings, travel):
        assert "by month" in charts.history_chart(rows, view) or view is everything


def test_subcategory_colors_are_shades_of_their_category():
    user_budget = budget.empty_budget()
    colors = design.subcategory_colors(user_budget)
    assert colors["necessities"] == design.CATEGORY_COLORS["needs"]  # alone in its category: the colour itself
    assert colors["discretionary"] != colors["travel"]
    assert design.marker("wants").count("Wants") == 1


def test_fill_sample_months_only_fills_empty_earlier_months():
    user_budget = make_budget()  # October has real data
    assert budget.fill_sample_months(user_budget, "2026-11") == 4
    assert budget.get_month(user_budget, "2026-10")["income"][0]["name"] == "Salary"
    assert "2026-11" not in user_budget["months"]
    assert all(item["name"].endswith(budget.SAMPLE_TAG) for item in budget.get_month(user_budget, "2026-06")["income"])
    assert budget.fill_sample_months(user_budget, "2026-11") == 0
    assert budget.months_with_data(user_budget) == 5
    assert budget.tiers(user_budget) == ["Retirement", "Brokerage"]
    assert budget.month_history(user_budget, "2026-07", 1)[0]["invested"] == 40000


# ---------- storage.py ----------

@pytest.fixture(autouse=True)
def temp_data_dir(tmp_path, monkeypatch):
    monkeypatch.setattr(storage, "DATA_DIR", str(tmp_path))


@pytest.mark.parametrize("typed, key", [("Mark", "mark"), (" Mark  G ", "mark_g"), ("../../etc/x", "etc_x"), ("Ana-María", "ana-mar_a")])
def test_any_name_maps_to_a_safe_file_key(typed, key):
    assert storage.file_key(typed) == key


@pytest.mark.parametrize("typed, message", [("", "Enter your name"), ("   ", "Enter your name"), ("!!!", "letter or number"), ("x" * 31, "30 characters")])
def test_unusable_names_are_rejected(typed, message):
    with pytest.raises(ValueError, match=message):
        storage.file_key(typed)


def test_new_name_starts_with_an_empty_budget():
    assert storage.load_budget("nobody") == budget.empty_budget()


def test_budget_round_trips_through_a_file():
    user_budget = make_budget()
    user_budget["settings"]["savings_goal"] = 500000
    storage.save_budget("mark", user_budget)
    assert storage.load_budget("mark") == user_budget
    assert storage.load_budget("someone_else") == budget.empty_budget()


def test_delete_budget(tmp_path):
    storage.save_budget("mark", make_budget())
    storage.delete_budget("mark")
    assert not (tmp_path / "mark.json").exists()
    storage.delete_budget("mark")  # deleting twice is fine


def test_first_sign_in_sets_the_password_and_later_ones_check_it():
    key, user_budget = storage.sign_in(" Mark ", "correct horse")
    assert key == "mark" and user_budget["account"]["name"] == "Mark"
    assert "correct horse" not in str(user_budget["account"])
    assert storage.sign_in("mark", "correct horse")[0] == "mark"
    with pytest.raises(ValueError, match="don't match"):
        storage.sign_in("Mark", "wrong password")


def test_budget_saved_before_passwords_takes_the_next_one_typed():
    storage.save_budget("mark", make_budget())
    key, user_budget = storage.sign_in("Mark", "new password")
    assert budget.section_total(budget.get_month(user_budget, "2026-10"), "income") == 320000
    assert storage.check_password(storage.load_budget(key)["account"], "new password")


def test_short_passwords_are_refused():
    with pytest.raises(ValueError, match="at least 8"):
        storage.sign_in("Mark", "short")
    assert storage.load_budget("mark") == budget.empty_budget()  # nothing saved


def test_same_password_gets_a_different_salt():
    assert storage.hash_password("same one") != storage.hash_password("same one")


@pytest.mark.parametrize("typed, email", [("", ""), ("  a@b.co ", "a@b.co")])
def test_clean_email_accepts(typed, email):
    assert storage.clean_email(typed) == email


@pytest.mark.parametrize("typed", ["nope", "a@b", "a b@c.d", "x" * 250 + "@b.co"])
def test_clean_email_rejects(typed):
    with pytest.raises(ValueError, match="name@example.com"):
        storage.clean_email(typed)


def test_rename_budget_refuses_a_taken_name(tmp_path):
    storage.save_budget("mark", make_budget())
    storage.save_budget("ana", budget.empty_budget())
    with pytest.raises(ValueError, match="taken"):
        storage.rename_budget("mark", "ana")
    storage.rename_budget("mark", "marcus")
    assert not (tmp_path / "mark.json").exists() and storage.load_budget("marcus") == make_budget()


def test_limit_meters_redden_as_they_fill():
    assert design.limit_color(0) == design.ACCENT
    assert design.limit_color(budget.WARN_AT) == design.WARN
    assert design.limit_color(1.0) == design.limit_color(1.5) == design.OVER
    halfway = design.limit_color(0.5)
    assert halfway not in (design.ACCENT, design.WARN, design.OVER)


## Run the tests

Every line should pass.

In [ ]:
!python -m pytest -q test_budget.py

## Run the app

In [ ]:
# Start the app. Run this cell again after changing any file above.
import re
import subprocess
import sys
import time

STARTUP_SECONDS = 90  # the public link can take a while to appear

if "server" in globals() and server.poll() is None:
    server.terminate()  # stop the copy started earlier in this session
    server.wait()

log = open("app.log", "w")
server = subprocess.Popen([sys.executable, "-u", "app.py", "--share"], stdout=log, stderr=subprocess.STDOUT)
for _ in range(STARTUP_SECONDS):
    time.sleep(1)
    started = open("app.log").read()
    if "public URL" in started or "share link" in started.lower() or server.poll() is not None:
        break
print(started)
if server.poll() is not None:
    raise RuntimeError("The app stopped while starting. The error is printed above.")

port = re.search(r"127\.0\.0\.1:(\d+)", started)
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(int(port.group(1)) if port else 7860, height=900)
except ImportError:
    print("Not in Colab: open the local or public URL above.")

In [ ]:
# If a button shows an error, run this and send the output: it is the app's own log.
!tail -n 60 app.log